# On-policy distillation

> **Historical Qwen3 experiment.** This notebook retains the original models and
> checkpoint lineage. The new Qwen3.5 student and Qwen3.8 teacher are configured in
> [Notebook 05](05_qwen38_restart.ipynb). Do not resume these checkpoints with new models.

**Notebook 04 · Diagnose stopping behavior and compare two learning signals**

We compare continued supervised practice with frozen-teacher feedback, starting
from the same verified Qwen3 student checkpoint. Every response, token boundary,
weight update, and evaluation has an explicit role in the experiment.

| Research question | Can teacher feedback improve the supervised student? |
| :--- | :--- |
| Student / teacher | `Qwen3-0.6B-Base` / frozen `Qwen3-32B` |
| Method | On-policy, full-vocabulary forward KL |
| Teacher context | Pinned non-thinking chat template |
| Next GPU milestone | Matched 32-update raw-token and semantic-STOP pilots |
| Evidence | Two recorded stopping failures, teacher probes, and a completed SFT control |
| Current limit | Revised safeguards and new-model GPU qualification remain pending |

**Read in order** · [1 Goal](#goal) · [2 Runtime](#setup) · [3 Data](#data) ·
[4 Loss](#loss) · [5 Starting point](#starting) · [6 Control](#control) ·
[7 Teacher](#teacher) · [8 Pilot](#training) · [9 Results](#results) · [10 Export](#save)

![The on-policy feedback loop](assets/04_on_policy_distillation/on_policy_workflow.png)

*The student writes the response. Each model reads that same response prefix in
its own question context. Only student parameters are updated.*

**Current status.** The collected Amarel EOS comparison is locally audited. Raw-token
training health-paused at update 18. Grouped STOP completed 32 updates and failed
quality gates. Both arms already lost strict accuracy at update 8. The revised
guarded policy is CPU-tested and awaits a new GPU experiment.


### Check the notebook connection

Run the next two short cells to identify the machine behind the selected kernel.
The saved outputs below record the user's Amarel connection on November 15, 2026:
`gpuk011`, two L40S GPUs, and both 03 archives present. Rerun them when reconnecting;
a saved output does not prove that your current connection is still active.

For this allocation, Jupyter listens on remote port `8888`. The laptop SSH tunnel
forwards local port `8890` to that port. Select the existing server in VS Code
using `http://127.0.0.1:8890/?token=YOUR_TOKEN`, with your private Jupyter token.
Keep the remote Jupyter process and the laptop tunnel running.


In [3]:
# Check which remote machine executes this notebook, independently of its saved outputs.
# CUDA must be available in the allocated GPU kernel before running the experiment.
import socket

import torch

print("Host:", socket.gethostname())
print("CUDA available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())

Host: gpuk011.amarel.rutgers.edu
CUDA available: True
Visible GPUs: 2


In [4]:
# Inspect the GPUs assigned to this kernel and the saved 03 archives on Amarel.
# These checks confirm access only; they do not load models or evaluate the EOS pilots.
import socket
from pathlib import Path

import torch

print("Execution host:", socket.gethostname())
print("Visible GPUs:", torch.cuda.device_count())

for index in range(torch.cuda.device_count()):
    print(f"GPU {index}:", torch.cuda.get_device_name(index))

project_root = Path.home() / "on-policy-distillation"
for filename in ("checkpoint.zip", "report.zip"):
    archive = project_root / "artifacts" / "sft03" / filename
    print(filename, "FOUND" if archive.is_file() else "MISSING")

Execution host: gpuk011.amarel.rutgers.edu
Visible GPUs: 2
GPU 0: NVIDIA L40S
GPU 1: NVIDIA L40S
checkpoint.zip FOUND
report.zip FOUND


---
<a id="goal"></a>
## 1. Understand the experiment

Think of a student solving a math problem on paper. In 03, we supplied a worked
solution and taught the student to predict its words. Here, the student writes
its **own** solution. After each partial answer, the teacher tells us how likely
each possible next token should be. A token is a small piece of text, sometimes
a whole word and sometimes only part of one.

The teacher is **frozen**, so its weights never change. Only the small student learns.
The teacher's probabilities are guidance about text, not a guarantee that the
arithmetic is correct. A teacher can still prefer a wrong continuation.

| Starting point | Additional practice | Purpose |
| :--- | :--- | :--- |
| **Starting SFT** | None | Measure the checkpoint we already have |
| **Continued SFT** | 1,024 new reference solutions | Check what more ordinary supervised practice achieves |
| **On-policy KD** | The same 1,024 questions, student-written responses, teacher distributions | Test learning on prefixes the student actually visits |

The full training recipe starts both arms from **identical checkpoint bytes**, uses
the same question order, and makes 128 optimizer updates with eight examples per
update. The next EOS comparison stops each fresh KD pilot after at most 32 updates. Their token
counts, compute cost, and learning signals differ. This comparison measures the two methods together with their different learning
signals and token budgets. Isolating the sampling policy would require another arm.

![Three arms from the same checkpoint](assets/04_on_policy_distillation/controlled_comparison.png)

### How this relates to the paper

We implement the fully on-policy case of [GKD, Section 3.1](https://arxiv.org/html/2306.13649v3#S3.SS1):
sample from the current student, hold those token IDs fixed, and match the teacher's
next-token distributions on those prefixes. We use **forward KL** and teacher
temperature `0.1`, following its arithmetic experiment choices. Our Qwen models,
reference-solution warm start, prompt, data budget, and training scale are different.
This is an educational adaptation, not the paper's reported benchmark experiment.

**On-policy** describes where responses come from, not the direction of KL.
01 used reverse KL to inspect mechanics. 04 uses forward KL for this experiment.
Teacher-generated answers are used only to measure teacher validation performance.
Training targets come from teacher distributions on student-written prefixes.
This experiment uses full student fine-tuning with no answer filtering.


### 1.1 What the first run taught us

The original shared plain-text prompt produced a misleading learning signal.
Detailed logs contain 52 updates. Every response from updates 24 through 52 hit
the 256-token cap, with examples that repeat words or restart the question.
Reference-token validation loss increased even while fixed-prefix KL decreased.

A user-run A100 probe scored the **same early response** under two teacher contexts.
At its final position, combined stopping probability changed from about
$3.93\times10^{-8}$ with the raw prompt to **0.530** with non-thinking chat
formatting. At target temperature 0.1, chat stopping mass was approximately **0.777**.
The observed student EOS itself stayed unlikely. The teacher preferred its own
`im_end` token. Both token IDs are valid stopping events for our student.

This single probe supports a teacher-context explanation. It does not establish
that a corrected training run will improve accuracy. The example response was
itself arithmetically wrong, illustrating why stopping and correctness need
separate measurements.

![Observed failure and teacher stopping probe](assets/04_on_policy_distillation/observed_failure.png)

*Measured evidence from the interrupted experiment and one response-boundary probe.
The dashed response-length ceiling is the configured generation budget.*

The [archived evidence](assets/04_on_policy_distillation/recorded_failure/manifest.json)
and [investigation](assets/04_on_policy_distillation/INVESTIGATION.md) preserve
this failure. Later console output reaches update 61, while complete downloaded
trajectories and history cover only updates 1 through 52.

### The follow-up Amarel pilot

With non-thinking teacher context, the teacher reached **52/64 strict validation
matches**, compared with **32/64** for the starting student and **40/64** for the
completed continued-SFT control. Alignment checks passed. During distillation,
responses became longer and more repetitive. The guard paused at update 22,
when **24 of the latest 32 responses** reached the 256-token cap. SLURM then ended
the allocation during follow-up validation. Reference loss had risen from 0.667
to 0.819 at update 16 while fixed-prefix KL fell.

The next hypothesis concerns how the loss interprets ending tokens. The student
learned `endoftext`, while earlier teacher probes preferred `im_end`. Both end
this single-turn response, but raw-token KL still compares their probabilities
separately. The next controlled comparison changes only that interpretation.
Its results will test how much ending-token disagreement contributes to the failure.
[Recorded Amarel evidence](assets/04_on_policy_distillation/recorded_amarel_pilot/README.md)

### The completed EOS comparison

The audited follow-up started at 33/64 strict matches and reference NLL
0.668. Raw-token training health-paused at update 18 with 1/64 matches. Grouped
STOP completed 32 updates with 23/64 matches and NLL 0.964. Both validations
finished before the allocation expired. The grouped arm failed the review gates.

![User-reported validation endpoints from the EOS comparison](assets/04_on_policy_distillation/pilot_outcomes.png)

*These endpoints come from the collected Amarel records. All 400 training trajectories and saved
validation scores passed local replay. The final checkpoints have different update counts.*

The stopping-token correction reduced one source of disagreement. Remaining
accuracy loss requires inspecting intermediate responses and measurements. The
latest local Downloads histories belong to an older 52-update run.

### The safeguard gap in the earlier code

The earlier guard paused only when 24 of the latest 32 sampled responses were
capped. A reference-loss increase produced a warning, and health was checked
before the newest validation measurements were recorded. The grouped pilot could
therefore continue while its answer quality deteriorated.

The revised policy checks the newest validation before another update. It pauses
when reference NLL rises over 10%, strict accuracy falls over five percentage
points, or more than 8 of the latest 32 responses hit the token limit. Each
threshold is an engineering review setting. Its configuration is signed into the
new experiment and saved with recovery metadata.

### Where the deterioration first appears

![Verified validation history and sampled-response caps](assets/04_on_policy_distillation/pilot_history.png)

*Points show saved GPU measurements. The dotted line marks update 8, the first
recorded validation. The new quality guard is replayed against this history.*

At update 8, raw-token strict matches fell from 33/64 to 17/64. Grouped STOP fell
to 29/64, with five previously wrong answers becoming correct and nine previously
correct answers becoming wrong. Reference NLL was still close to baseline at
0.679 and 0.673, respectively. A guard that only watches NLL would miss this early
accuracy decline. The selected accuracy limit requests a pause at update 8 in both arms.

One grouped-stop regression changes a correct staircase calculation of 15 + 9 = 24
into an invented extra staircase term and concludes 25. Another changes a correct
40-mile route into 20 miles. Both use the requested final-answer marker. Arithmetic
and format failures both occur in the recorded responses.

The fixed-prefix grouped KL falls from 0.918 to about 0.646 by the grouped arm's
last checkpoint while reference NLL rises to 0.964. Those objectives measure
agreement on different prefixes and targets. The logged gradient norms remain
finite and clipping is active. These records support a learning-quality failure.
The contribution of teacher sharpness, update size, and student capacity requires
a controlled follow-up. New probes record entropy and ordinary/EOS loss components
so those hypotheses can be examined directly.


---
<a id="setup"></a>
## 2. Prepare the runtime and saved checkpoint

The default setup runs a **fresh EOS comparison** from your saved 03 checkpoint.
Your current Amarel connection checks already locate both input archives. No new
upload or rerun of Notebook 03 is needed while those remote files remain available.

| Setting | Value for this run | Meaning |
| :--- | :--- | :--- |
| `MODE` | `"train"` | Load the real models and perform the experiment |
| `RUN_STAGE` | `"eos_comparison"` | Compare two ways to interpret ending tokens |
| `EOS_LOSS_MODE` | `"raw_token"` | Default outside the comparison; the comparison runs both modes |
| `RESUME_RUN_DIRECTORY` | `None` | Start a new run rather than resuming one |
| `HISTORICAL_RUN_DIRECTORY` | `None` | Use only the verified 03 inputs, without old 04 snapshots |

Each pilot starts from the same 03 weights and pauses after at most 32 updates.
The original 03 archives remain unchanged. For a CPU-only check of data, token
boundaries, and the math, set `OPD_TRAIN_MODE=inspect` before starting the kernel.
The local validator uses that separate inspection setting automatically.

**Run in order.** Execute Sections 2 through 7, including the starting-student
measurements, frozen-teacher checks, and Section 7.1 stopping probes. Confirm
that these finish without errors before starting Section 8. Sections 9 and 10
then summarize and export the comparison; the final held-out test remains reserved.

Copy the verified Notebook 03 archives to the remote runtime before running.
Colab uses `/content/checkpoint.zip` and `/content/report.zip`. Amarel uses
`artifacts/sft03/checkpoint.zip` and `artifacts/sft03/report.zip` inside the project.
Files in your laptop's Downloads folder must be transferred to the server first.

The BF16 teacher loader supports **one almost-empty 80GB GPU or two almost-empty
GPUs with at least 40GB of nominal memory each**, including this two-L40S allocation. On two cards, the student stays on GPU 0 while the frozen teacher's
layers are divided across both GPUs. This is model sharding. GPU memory remains
separate, and activations travel between cards. The teacher's model, precision,
and loss are unchanged. CPU/disk weight offload is rejected.

The loader reserves weight headroom for student training, then plans whole
teacher-layer placement using empty tensors. This is an estimate. Run the
one-update smoke test described in the next-run checklist before the full
notebook, especially on a new hardware layout. A successful short response does
not establish the memory requirement of every 1,024-token sequence.

On Amarel, Jupyter must run inside your SLURM allocation, with both assigned GPUs
visible to PyTorch. Keep SLURM's `CUDA_VISIBLE_DEVICES` unchanged. Install project
dependencies into your user environment and start Jupyter in the project root.
The notebook checks device memory and disk before loading weights. It preserves
the installed PyTorch/CUDA. The first teacher download is about 65.5 GB, so check
your storage quota and allocation time. Use a fresh run for the new code/layout.

### Runtime and cache setup

Restart the kernel, then run the setup cell below. It finds the project from the
working directory or `~/on-policy-distillation`, and sets the cache paths before
Hugging Face libraries are imported. A separate scratch-cache cell is unnecessary.

| Files | Amarel location |
| :--- | :--- |
| Project, 03 archives, and new experiment results | `~/on-policy-distillation` |
| Downloaded model and dataset cache | `/scratch/<username>/huggingface` |

Scratch is temporary, unbacked-up storage. Downloaded models can be fetched again.
Keep reports and checkpoints in persistent storage and retain your local backups.
The cell respects explicit cache environment variables. To switch from an old
cache, restart the kernel and set the intended paths before running the setup.
Local inspection and Colab default to the project cache.

### Connect and run

In VS Code, select your allocated Amarel Jupyter kernel or Colab kernel and run from the top. The full project must
also exist on that remote server. A local notebook tab does not automatically
upload its `src/` helpers. Dependency changes require one kernel restart.
Restart the Python kernel before training to free GPU memory held by models
left over from 03. Saved runtime files remain available after a kernel restart.
The two EOS pilots each stop after at most 32 distillation updates. Save their report and
resume checkpoints before disconnecting. Request a six-hour allocation for this
comparison. The deadline guard reserves ten minutes for saving state. Each completed
validation answer is saved immediately, so interrupted evaluation can continue.

To resume an incomplete comparison, set `RESUME_RUN_DIRECTORY` to its parent run
folder and keep `RUN_STAGE = "eos_comparison"`. After a passing reviewed comparison,
set `EOS_LOSS_MODE = "semantic_stop"`, `RUN_STAGE = "full"`, and the resume path to
that run's `eos_comparison/semantic_stop` folder. The optimizer, scheduler, and RNG
continue from its 32-update boundary.

If another notebook kernel holds GPU memory, restarting this notebook alone may
leave that other process alive. Compare the printed kernel PID with `nvidia-smi`
in the allocated runtime's terminal. A model-loading memory guard explains insufficient space
before allocating another teacher. A Hub-secret warning can occur through VS Code.
Public model calls use `token=False` and work without a personal token.

### Observed two-GPU smoke test

On November 8, 2026, the user completed one disposable update on two A100 40GB
GPUs. The student generated 51 response tokens and stopped with EOS. The recorded
KL was 1.964 nats, with peak allocated memory of 36.18 GiB and 36.01 GiB. This
qualifies that short response and update. The subsequent Amarel pilot also completed
teacher validation and 22 updates, then exposed a stopping failure. Maximum-length
memory behavior and the new controlled EOS comparison remain unverified. The smoke
response was incorrect, so it supplies hardware and mechanics evidence only.

**Additional user-run hardware evidence, November 15, 2026.** On two L40S GPUs
(44.4 GiB visible per card), the disposable test reported
`one_disposable_update_passed`: 51 response tokens, one EOS stop, no token-limit
stop, finite raw-token loss `1.9868708849`, and learning rate `5e-6`. The shared
terminal excerpt does not include peak-memory measurements. This is evidence
for one short response and update, not a completed comparison or improved accuracy.
The test saved no new model weights.

For a future Colab session, follow the [Colab run guide](assets/04_on_policy_distillation/COLAB_RUN.md).


In [5]:
import importlib.metadata
import importlib.util
import os
import subprocess
import sys
import tempfile
from pathlib import Path

# Default to the corrected pilot on an allocated CUDA runtime.
# Choose "inspect" for a CPU-only check without loading model weights.
# The validator sets OPD_TRAIN_MODE=inspect in its separate local kernel.
MODE = os.environ.get("OPD_TRAIN_MODE", "train")
if MODE not in {"inspect", "train"}:
    raise ValueError("Choose MODE='inspect' or MODE='train'.")
IN_COLAB = bool(importlib.util.find_spec("google") and importlib.util.find_spec("google.colab"))
PROJECT_ROOT = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents, Path.home() / "on-policy-distillation"]
        if (p / "src/baseline.py").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    if not IN_COLAB:
        raise RuntimeError("Open Jupyter inside the project checkout.")
    PROJECT_ROOT = Path("/content/on-policy-distillation")
    if not PROJECT_ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "dev",
                "https://github.com/igoldin670/on-policy-distillation.git",
                str(PROJECT_ROOT),
            ],
            check=True,
        )
if not (PROJECT_ROOT / "src/opd.py").is_file():
    raise RuntimeError(
        "The remote checkout predates 04. Sync/upload the complete 04 project to this server, then restart the kernel."
    )

if IN_COLAB:
    # Use uv for dependencies. pip is used only to bootstrap uv if it is absent.
    # Never install a CPU torch wheel into the CUDA-enabled Colab environment.
    packages = [
        line.strip()
        for line in (PROJECT_ROOT / "requirements-generation.txt").read_text().splitlines()
        if line.strip() and not line.lstrip().startswith("#")
    ]
    packages += [
        line.strip()
        for line in (PROJECT_ROOT / "requirements-notebook.txt").read_text().splitlines()
        if line.split("==")[0] in {"numpy", "pandas", "matplotlib"}
    ]
    packages += ["bottleneck==1.5.0", "ipywidgets==8.1.8", "PyYAML==6.0.3"]
    changed = []
    for requirement in packages:
        name, version = requirement.split("==")
        try:
            installed = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            installed = None
        module_name = {"PyYAML": "yaml"}.get(name, name)
        loaded = sys.modules.get(module_name)
        if loaded is not None and getattr(loaded, "__version__", installed) != installed:
            raise RuntimeError(f"Restart the kernel: {name} has mixed loaded/installed versions.")
        if installed != version:
            changed.append(requirement)
    if changed:
        if importlib.util.find_spec("uv") is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        subprocess.run(
            [sys.executable, "-m", "uv", "pip", "install", "--python", sys.executable, *packages],
            check=True,
        )
        raise RuntimeError("Dependencies installed. Restart the kernel and run from the top.")

# Configure cache paths BEFORE importing transformers, datasets, or huggingface_hub.
# On Amarel, large downloadable model files belong in your scratch directory.
# Project code, input archives, and results stay in the project checkout.
scratch_directory = Path("/scratch") / os.environ.get("USER", "")
IN_AMAREL = bool(
    os.environ.get("SLURM_JOB_ID") and os.environ.get("USER") and scratch_directory.is_dir()
)
default_cache = (
    scratch_directory / "huggingface" if IN_AMAREL else PROJECT_ROOT / ".cache/source-verification"
)
# Preserve explicit cache overrides, including those supplied by local QA.
os.environ.setdefault("HF_HOME", str(default_cache))
model_cache = Path(os.environ["HF_HOME"])
model_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HUB_CACHE", str(model_cache / "hub"))
os.environ.setdefault("HF_DATASETS_CACHE", str(model_cache / "datasets"))
# Public pinned downloads do not need a Hugging Face secret or Google Drive.
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "opd-matplotlib"))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
# Jupyter may start in your home directory. Relative project paths need this root.
os.chdir(PROJECT_ROOT)
runtime_project_label = PROJECT_ROOT if MODE == "train" else PROJECT_ROOT.name
print(f"Mode: {MODE} | Colab: {IN_COLAB} | Runtime project: {runtime_project_label}")
print("Model cache:", model_cache)

Mode: train | Colab: False | Runtime project: /cache/home/igg11/on-policy-distillation
Model cache: /scratch/igg11/huggingface


In [6]:
import hashlib
import json
import math
import time
from dataclasses import asdict

import pandas as pd
import torch
import yaml
from datasets import disable_progress_bars, load_dataset
from IPython.display import display
from transformers import AutoConfig, AutoTokenizer, get_linear_schedule_with_warmup

from src.baseline import verify_recorded_run
from src.data import prepare_opd_splits
from src.generation import sample_trajectory
from src.logprobs import (
    response_prediction_logits,
    selected_log_probs,
    teacher_prompt_token_ids,
    verify_shared_tokenizer,
)
from src.opd import (
    TEACHER_ID,
    TEACHER_REVISION,
    OPDConfig,
    append_trajectories,
    backward_opd_group,
    backward_sft_group,
    cuda_preflight,
    experiment_signature,
    export_report,
    fixed_trajectory_diagnostics,
    load_student,
    load_teacher,
    new_run_directory,
    optimizer_update,
    paired_accuracy_difference,
    read_resume,
    release_cuda_memory,
    resolve_cold_start,
    restore_training_state,
    seed_everything,
    training_health,
    trajectory_tensors,
    truncate_uncommitted_trajectories,
    validate_trajectory_log,
)
from src.opd_diagnostics import load_historical_probes, probe_checkpoint_set
from src.opd_experiment import (
    SlurmBudget,
    check_comparison_storage,
    evaluate_incrementally,
    isolated_evaluation_rng,
    run_eos_comparison,
    verify_historical_run,
)
from src.opd_loss import forward_kl_loss, forward_kl_per_token
from src.opd_plots import plot_run
from src.sft import (
    SFTConfig,
    collate_examples,
    encode_example,
    evaluation_summary,
    load_checkpoint_tokenizer,
    precision_context,
    prepare_splits,
    save_checkpoint,
    validation_loss,
    verify_checkpoint,
)
from src.training_safety import TrainingSafetyConfig
from src.utils import sha256_file, verified_zip, verify_inventory, write_json

# Copy the matching Notebook 03 ZIPs to the runtime before training.
# Colab uses /content. On Amarel, put them in PROJECT_ROOT/artifacts/sft03.
# These are remote paths, separate from your laptop's Downloads directory.
SFT_ARTIFACT_DIRECTORY = Path("/content") if IN_COLAB else PROJECT_ROOT / "artifacts/sft03"
SFT_REPORT_PATH = SFT_ARTIFACT_DIRECTORY / "report.zip"
SFT_CHECKPOINT_PATH = SFT_ARTIFACT_DIRECTORY / "checkpoint.zip"

# Fresh comparison: keep None to start from the verified 03 checkpoint.
# To resume after interruption, set this to the existing 04 run directory.
# Matching optimizer-boundary snapshots are discovered for each arm separately.
RESUME_RUN_DIRECTORY = None

# The pilot pauses after 32 updates of the SAME 128-update experiment.
# After reviewing it, choose "full" and set RESUME_RUN_DIRECTORY above.
RUN_STAGE = "eos_comparison"
# The comparison constructs BOTH raw_token and semantic_stop pilots itself.
# Leave this default unchanged for the fresh comparison; do not run two notebooks.
EOS_LOSS_MODE = "raw_token"
# A fresh comparison requires only the verified 03 archives. To inspect an old
# run as well, set this to its complete results directory on the current server.
HISTORICAL_RUN_DIRECTORY = None
if RUN_STAGE not in {"eos_comparison", "pilot", "full"}:
    raise ValueError("Choose eos_comparison, pilot, or full.")
if MODE == "train" and RUN_STAGE == "full" and RESUME_RUN_DIRECTORY is None:
    raise ValueError("Run and review the pilot first, then set its RESUME_RUN_DIRECTORY.")

# Versioned safeguards define a NEW experiment identity. Existing saved outputs
# describe the historical policy. None selects historical warning-only behavior.
safety_policy = TrainingSafetyConfig()
config = OPDConfig(eos_loss_mode=EOS_LOSS_MODE)
seed_everything(config.seed)
# Fixed seeds make random selections/draws repeatable on the same setup.
# Different GPU/software versions can still produce numerical differences.
disable_progress_bars()
baseline_evidence = verify_recorded_run(
    PROJECT_ROOT / "notebooks/assets/02_baseline_evaluation/recorded_run"
)
protocol = baseline_evidence["manifest"]["protocol"]
model_choices = yaml.safe_load((PROJECT_ROOT / "configs/qwen3_legacy.yaml").read_text())
assert model_choices["student_model_id"] == protocol["student_model_id"]
assert model_choices["teacher_model_id"] == TEACHER_ID
lineage = json.loads(
    (
        PROJECT_ROOT / "notebooks/assets/04_on_policy_distillation/cold_start_lineage.json"
    ).read_text()
)
if MODE == "train":
    # Print the selected recipe before slow archive verification or model loading.
    # This makes it easy to spot an accidental resume path or inspection setting.
    print(
        "Active run settings:",
        {
            "mode": MODE,
            "stage": RUN_STAGE,
            "eos_modes": ["raw_token", "semantic_stop"]
            if RUN_STAGE == "eos_comparison"
            else [EOS_LOSS_MODE],
            "resume_directory": str(RESUME_RUN_DIRECTORY)
            if RESUME_RUN_DIRECTORY is not None
            else None,
            "historical_directory": str(HISTORICAL_RUN_DIRECTORY)
            if HISTORICAL_RUN_DIRECTORY is not None
            else None,
        },
        flush=True,
    )
    print("03 checkpoint archive:", SFT_CHECKPOINT_PATH, flush=True)
    print("03 report archive:", SFT_REPORT_PATH, flush=True)
    print("Verifying 03 inputs and checking the GPU allocation...", flush=True)
    # A verified report is not a substitute for the actual trained student weights.
    sft_report_directory, starting_checkpoint_directory, sft_evidence = resolve_cold_start(
        PROJECT_ROOT, report_path=SFT_REPORT_PATH, checkpoint_path=SFT_CHECKPOINT_PATH
    )
    assert sft_evidence["protocol"] == protocol, "Keep 02/03's exact source and prompt protocol."
    runtime = cuda_preflight(PROJECT_ROOT)
    run_budget = SlurmBudget.from_environment()
    run_budget.require_time()
    if run_budget.end_time is not None:
        print("Allocation minutes remaining:", round((run_budget.end_time - time.time()) / 60, 1))
        print("Checkpoint reserve minutes:", run_budget.reserve_seconds / 60)
    # Respect the CPU count granted to this SLURM task for tensor work.
    if os.environ.get("SLURM_CPUS_PER_TASK"):
        torch.set_num_threads(int(os.environ["SLURM_CPUS_PER_TASK"]))
    tokenizer = load_checkpoint_tokenizer(starting_checkpoint_directory)
    cold_splits = sft_evidence["splits"]
    checkpoint_hashes = sft_evidence["checkpoint_hashes"]
    print("Kernel PID:", runtime["kernel_pid"], "| stage:", RUN_STAGE)
    print("Verified starting checkpoint:", starting_checkpoint_directory)
    print("GPU:", runtime["gpu"], "| memory GiB:", round(runtime["gpu_total_gib"], 1))
    print("BF16 supported:", runtime["bf16_supported"])
    print("Teacher layout:", runtime["teacher_layout"])
    display(pd.DataFrame(runtime["gpus"]))
else:
    # Reconstruct the saved 03 split with real pinned data; no Qwen weights load.
    tokenizer = AutoTokenizer.from_pretrained(
        protocol["student_model_id"], revision=protocol["student_revision"], token=False
    )
    checkpoint_hashes = lineage["checkpoint_hashes"]
    print("INSPECT: tokenizer/data/math only. No teacher or student weights loaded.")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Fixed recipe:", asdict(config))
print("Active safety policy:", asdict(safety_policy) if safety_policy is not None else "historical")
print("Optimizer updates per arm:", config.optimizer_steps)
print("Teacher revision:", TEACHER_REVISION)

Active run settings: {'mode': 'train', 'stage': 'eos_comparison', 'eos_modes': ['raw_token', 'semantic_stop'], 'resume_directory': None, 'historical_directory': None}
03 checkpoint archive: /cache/home/igg11/on-policy-distillation/artifacts/sft03/checkpoint.zip
03 report archive: /cache/home/igg11/on-policy-distillation/artifacts/sft03/report.zip
Verifying 03 inputs and checking the GPU allocation...
Allocation minutes remaining: 310.3
Checkpoint reserve minutes: 10.0
Kernel PID: 2535206 | stage: eos_comparison
Verified starting checkpoint: /cache/home/igg11/on-policy-distillation/.cache/opd-inputs/4ecf79a79f32e89944a606c41b79ce52d65841c151d25f5ead016648cd730eb6
GPU: NVIDIA L40S | memory GiB: 44.4
BF16 supported: True
Teacher layout: two_gpu_bf16


,index,name,total_gib,free_gib,teacher_weight_budget_gib
0,0,NVIDIA L40S,44.392151,43.968079,31.968079
1,1,NVIDIA L40S,44.392151,43.968079,40.968079


Fixed recipe: {'seed': 42, 'train_examples': 1024, 'test_examples': 256, 'accumulation_steps': 8, 'learning_rate': 5e-06, 'weight_decay': 0.01, 'max_grad_norm': 1.0, 'warmup_fraction': 0.1, 'validation_interval': 32, 'checkpoint_interval': 32, 'max_prompt_length': 512, 'max_response_length': 256, 'max_sequence_length': 1024, 'teacher_temperature': 0.1, 'kl_chunk_size': 16, 'diagnostic_examples': 8, 'teacher_context': 'qwen3_nonthinking_v1', 'eos_loss_mode': 'raw_token', 'eos_token_ids': (151643, 151645), 'pilot_updates': 32, 'pilot_check_interval': 8, 'cap_window_responses': 32, 'cap_pause_fraction': 0.75, 'nll_warning_ratio': 1.25}
Optimizer updates per arm: 128
Teacher revision: 9216db5781bf21249d130ec9da846c4624c16137


---
<a id="data"></a>
## 3. Keep practice, progress checks, and the exam separate

We use 1,024 **new** GSM8K training questions. They exclude every 03 practice and
validation question, and normalized question-text overlap with the official test
split. Continued SFT sees their cleaned reference solutions. On-policy KD receives
only their question prompts when it generates responses.

The same 64 validation questions from 03 are progress checks. The primary exam
contains 256 official test questions that were **not** in the 32-question subset
already inspected in 02/03. That earlier subset is reported separately as historical
context. We evaluate fixed final checkpoints, with no checkpoint selection or
hyperparameter tuning against either test set.

The candidate order and every exclusion are recorded. Long reference solutions
are skipped before training, never cut off mid-answer. Test selection uses the
question and prompt length, not how easy the answer looks.


In [7]:
# The TRAIN and TEST source revisions are pinned. Downloading data is not training.
source_train = load_dataset(
    protocol["dataset_id"],
    protocol["dataset_config"],
    split="train",
    revision=protocol["dataset_revision"],
    token=False,
)
source_test = load_dataset(
    protocol["dataset_id"],
    protocol["dataset_config"],
    split="test",
    revision=protocol["dataset_revision"],
    token=False,
)
if MODE == "inspect":
    cold_splits = prepare_splits(
        source_train, tokenizer, protocol, SFTConfig(**lineage["configuration"])
    )
    assert [r["source_index"] for r in cold_splits["train"]] == lineage["train_source_indices"]
    assert [r["source_index"] for r in cold_splits["validation"]] == lineage[
        "validation_source_indices"
    ]

# Check the saved text-to-token boundaries again with the tokenizer used NOW.
# This also detects tokenizer regex changes that vocabulary-size checks miss.
for record in cold_splits["train"] + cold_splits["validation"]:
    source_record = source_train[record["source_index"]]
    assert source_record["question"] == record["question"]
    encoded = encode_example(tokenizer, record["question"], source_record["answer"], protocol, 1024)
    assert encoded["input_ids"] == record["input_ids"]
    assert encoded["labels"] == record["labels"]
splits = prepare_opd_splits(
    source_train,
    source_test,
    tokenizer,
    protocol,
    cold_splits,
    train_examples=config.train_examples,
    test_examples=config.test_examples,
    seed=config.seed,
    max_prompt_length=config.max_prompt_length,
    max_sequence_length=config.max_sequence_length,
)
train_examples = splits["train"]
validation_examples = splits["validation"]
display(
    pd.DataFrame(
        [
            {"split": name, "questions": len(splits[name]), "role": role}
            for name, role in [
                ("train", "optimizer practice"),
                ("validation", "progress only"),
                ("primary_test", "final paired exam"),
                ("historical_test", "previously inspected context"),
            ]
        ]
    )
)
print("Selection exclusions recorded:", len(splits["exclusions"]))
print("One practice question:\n", train_examples[0]["question"])
print(
    "\nIts cleaned reference is used ONLY by the SFT control:\n", train_examples[0]["target_text"]
)

,split,questions,role
0,train,1024,optimizer practice
1,validation,64,progress only
2,primary_test,256,final paired exam
3,historical_test,32,previously inspected context


Selection exclusions recorded: 326
One practice question:
 Martin eats 1/2 cup of berries every day.  The grocery store is selling a package of berries (1 cup per pack) for $2.00.  How much will he spend on berries in a 30 day period?

Its cleaned reference is used ONLY by the SFT control:
 Martin eats 1/2 cup berries every day so over 30 days that's .5*30 = 15 cups of berries
The grocery store sells 1 cup packages of berries for $2.00 a pack.
Martin needs 15 cups so that's 2*15 = $30.00.
#### 30


### 3.1 What exactly receives a loss?

`input_ids` contains prompt tokens followed by response tokens. For SFT, the
response comes from the reference solution. For on-policy KD, it comes from the
student. Prompt positions provide context but have no direct training loss. Processing
that context still affects how the student predicts its response.

We process one unpadded response at a time during distillation. This removes the
need to guess which repeated end-token IDs are padding. SFT still uses explicit
`-100` labels to ignore prompt/padding positions. A real end token remains a target.

![The one-token prediction relationship](assets/04_on_policy_distillation/token_alignment.png)

If the prompt has `P` tokens, the first response token is at complete-sequence
index `P`, but its prediction is at logits index `P - 1`. The last prompt token's
logits predict what comes next. We therefore keep `logits[:, P - 1:-1, :]`.
The final logits position predicts an unobserved token and is excluded.


In [8]:
# A real reference example makes the boundary visible before any model is loaded.
example = train_examples[0]
prompt_length = example["prompt_length"]
preview_batch = collate_examples(train_examples[:2], tokenizer.pad_token_id)
print(
    "SFT tensors [batch_size, padded_sequence_length]:",
    {key: tuple(value.shape) for key, value in preview_batch.items()},
)
boundary_positions = range(prompt_length - 2, prompt_length + 3)
display(
    pd.DataFrame(
        [
            {
                "input_position": position,
                "role": "prompt" if position < prompt_length else "reference response",
                "token_id": example["input_ids"][position],
                "decoded_token": repr(tokenizer.decode([example["input_ids"][position]])),
                "training_label": example["labels"][position],
                "predicting_logits_position": position - 1,
            }
            for position in boundary_positions
        ]
    )
)
for row_index, record in enumerate(train_examples[:2]):
    complete_length = len(record["input_ids"])
    assert (preview_batch["labels"][row_index, : record["prompt_length"]] == -100).all()
    assert preview_batch["labels"][row_index, complete_length - 1].item() == tokenizer.eos_token_id
    assert (preview_batch["labels"][row_index, complete_length:] == -100).all()
print("Prompt masks, response boundaries, real EOS targets, and padding checks passed.")

SFT tensors [batch_size, padded_sequence_length]: {'input_ids': (2, 169), 'attention_mask': (2, 169), 'labels': (2, 169)}


,input_position,role,token_id,decoded_token,training_label,predicting_logits_position
0,78,prompt,16141,'Answer',-100,77
1,79,prompt,510,':\n',-100,78
2,80,reference response,49119,'Martin',49119,79
3,81,reference response,49677,' eats',49677,80
4,82,reference response,220,' ',220,81


Prompt masks, response boundaries, real EOS targets, and padding checks passed.


---
<a id="loss"></a>
## 4. Understand the learning signal

### 4.1 One answer token versus all possible next choices

Suppose a prefix ends with `3 + 2 =`. The following table uses a
**made-up three-token vocabulary**.

| Next token | Teacher's probability | Student's probability |
| :--- | ---: | ---: |
| `5` | 80% | 10% |
| `6` | 15% | 70% |
| `7` | 5% | 20% |

SFT would say, "the reference token is `5`, so make that token less surprising."
Forward KL instead compares both complete lists. The teacher strongly favors
`5`, while the student favors `6`. Learning should move the student's list toward
the teacher's. This is what **comparing possible next choices** means.

The real models have 151,936 output columns, not three. We keep all columns.
The sampled token's log probability is useful for inspection, but is **not**
the full-vocabulary KL loss.

### 4.2 The equation and the Python operation

For one student-written prefix, let $q$ be the teacher's target distribution and
$p_\theta$ the student's distribution. The temperature $T$ controls how concentrated the teacher target becomes.

$$
q(v)=\operatorname{softmax}(z_T/T)_v,\qquad
p_\theta(v)=\operatorname{softmax}(z_S)_v.
$$

$$
D_{\mathrm{KL}}(q\Vert p_\theta)
=\sum_{v\in\mathcal V} q(v)\big[\log q(v)-\log p_\theta(v)\big].
$$

`forward_kl_per_token` implements this expression directly using FP32
`log_softmax`, multiplication, and a vocabulary sum. A natural logarithm gives
units called **nats**. Zero means identical distributions. Small negative values
near zero can arise from floating-point rounding. Larger negatives are errors.

For a batch of $B$ sampled responses, response $i$ has $R_i$ tokens. The objective
averages positions within each response, then averages those response losses.

$$
\mathcal L_{\mathrm{OPD}}(\theta)
=\frac{1}{B}\sum_{i=1}^{B}\frac{1}{R_i}
\sum_{t=1}^{R_i} D_{\mathrm{KL}}\!\left(
q(\cdot\mid c_T(x_i),y_{i,<t})\Vert p_\theta(\cdot\mid c_S(x_i),y_{i,<t})\right).
$$

Here $c_S(x)$ is the student's original question prompt and $c_T(x)$ is the
teacher's non-thinking chat prompt for that question. The sampled response prefix
$y_{<t}$ is identical for both models. They have different question wrappers and
therefore different prompt lengths. This is an explicit adaptation of GKD.

First average across each response, then across responses, as in GKD's sequence
divergence. A long answer does not outweigh a short answer just by being longer.
Both 04 arms use this per-response weighting. 03 used a pooled token mean, which
is a different valid weighting and remains unchanged there.

The related soft-target cross-entropy is $H(q,p)=-\sum_v q(v)\log p(v)$.
Since $D_{\mathrm{KL}}(q\Vert p)=H(q,p)-H(q)$ and the teacher is frozen,
both have the same student gradient on fixed prefixes. We report **actual KL**,
including the entropy term, rather than calling cross-entropy KL.

### 4.3 Two temperatures, two separate jobs

- **Student sampling temperature = 1.0** draws responses from its current distribution.
  `top_k=0` and `top_p=1.0` remove truncation filters.
- **Teacher target temperature = 0.1** sharpens the teacher's choices for this
  arithmetic experiment. This does not control student sampling. The student
  loss distribution stays at temperature 1, with no extra $T^2$ multiplier.

We do not differentiate through the sampled token IDs. We run a new differentiable
student forward pass on their fixed prefixes. This is the GKD update, not a
policy-gradient estimate through the sampling process.
### 4.4 Two spellings of the same ending

Imagine both models agree there is an 80% chance the answer should end. One puts
that probability on `endoftext`, and the other on `im_end`. Raw-token KL penalizes
that difference. The grouped loss asks whether they agree on the decision to stop.
In this single-turn experiment, both ending IDs are accepted by the decoder.

![Illustration of two ending tokens grouped into one STOP event](assets/04_on_policy_distillation/eos_alignment.png)

*Illustrative probabilities. These numbers are separate from model measurements.*

Let $E$ contain the two ending IDs. Normalize the teacher with its temperature
first, then sum ending probabilities for both models.

$$
q_{\mathrm{stop}}=\sum_{e\in E}q(e),\qquad
p_{\mathrm{stop}}=\sum_{e\in E}p_\theta(e).
$$

The grouped forward KL at every response position is

$$
\sum_{v\notin E}q(v)\log\frac{q(v)}{p_\theta(v)}
+q_{\mathrm{stop}}\log\frac{q_{\mathrm{stop}}}{p_{\mathrm{stop}}}.
$$

`eos_token_ids=None` keeps the raw-token calculation. Passing the verified ending
IDs uses `logsumexp` over normalized log probabilities to calculate STOP mass.
All ordinary vocabulary contributions remain in the loss. We average response
positions and responses exactly as before. Original sampled IDs are preserved.
No synthetic EOS is added to a capped response.

The EOS-alignment paper studies sampled-token updates. This notebook applies its
semantic-action idea to full-distribution forward KL and tests that adaptation.
We report both KL definitions for both pilots because grouping changes the metric.
[EOS-alignment paper](https://arxiv.org/html/2609.20511v1)


In [9]:
# These three probabilities are an ILLUSTRATION, not measurements from Qwen.
# Use teacher temperature 1 here because the table already specifies q directly.
toy_teacher_probs = torch.tensor([[[0.80, 0.15, 0.05]]])  # [1, 1, 3]
toy_student_probs = torch.tensor([[[0.10, 0.70, 0.20]]])
toy_kl = forward_kl_per_token(toy_student_probs.log(), toy_teacher_probs.log(), 1.0)
manual_kl = (toy_teacher_probs * (toy_teacher_probs.log() - toy_student_probs.log())).sum(-1)
assert torch.allclose(toy_kl, manual_kl, atol=1e-6)
assert toy_kl.item() > 0
print(f"Illustration only: teacher-to-student forward KL = {toy_kl.item():.4f} nats.")
print("No model weights were loaded or updated by this example.")

Illustration only: teacher-to-student forward KL = 1.3632 nats.
No model weights were loaded or updated by this example.


---
<a id="starting"></a>
## 5. Freeze the protocol and inspect the starting student

The latest verified 03 run reached **60/64 format compliance**, **32/64 strict
matches**, and **30/64 exploratory prose matches** on validation. Reference NLL
fell from 0.776 to 0.667, while four responses reached the token limit. These
historical measurements describe the supervised starting point.

04 recognizes both `151643` (`endoftext`) and `151645` (`im_end`) as valid stops.
Their text meanings match across tokenizers, although the models' default EOS
roles differ. Paired student evaluations are rerun under this same stopping rule.
The teacher can assign ending probability to `im_end`. Grouping combines stopping
probabilities while preserving every original token ID.

Evaluation is greedy with a maximum of 512 new tokens. Training samples at temperature 1 with
a maximum of 256 new tokens. A capped response is retained as a truncated prefix and flagged.
We do not invent an EOS or discard a response based on its answer.

Teacher prompts use a pinned chat template with an empty thinking block. Student
prompts retain the original 03 format. The next cell verifies shared token meanings,
prepares teacher prefixes, and records the complete experiment signature.


In [10]:
# Reading teacher metadata/tokenizer does not download the 32B weight files.
teacher_config = AutoConfig.from_pretrained(TEACHER_ID, revision=TEACHER_REVISION, token=False)
teacher_tokenizer = AutoTokenizer.from_pretrained(
    TEACHER_ID, revision=TEACHER_REVISION, token=False
)
student_config = AutoConfig.from_pretrained(
    protocol["student_model_id"], revision=protocol["student_revision"], token=False
)
assert student_config.vocab_size == teacher_config.vocab_size == 151936
tokenizer_checks = verify_shared_tokenizer(
    tokenizer,
    teacher_tokenizer,
    [
        text
        for row in train_examples + validation_examples
        for text in (row["prompt"], row["target_text"])
    ]
    + ["  spaces\n\n", "3 + 2 = 5.\n#### 5", "£12.50 − 3", "Café 中文 🙂"],
    teacher_config.vocab_size,
)
# These prefix IDs are part of the experiment and each rollout audit record.
# Tokenize the teacher context ONCE, then append original response IDs unchanged.
for name in ("train", "validation"):
    splits[name] = [
        {
            **record,
            "teacher_prompt_token_ids": teacher_prompt_token_ids(
                teacher_tokenizer, record["prompt"]
            ),
        }
        for record in splits[name]
    ]
train_examples, validation_examples = splits["train"], splits["validation"]
for record in train_examples + validation_examples:
    assert (
        len(record["teacher_prompt_token_ids"]) + config.max_response_length
        <= config.max_sequence_length
    )
teacher_template_sha256 = hashlib.sha256(teacher_tokenizer.chat_template.encode()).hexdigest()

stop_token_ids = sorted({student_config.eos_token_id, teacher_config.eos_token_id})
assert tuple(stop_token_ids) == tuple(config.eos_token_ids), (
    "Verify ending semantics before grouping."
)
evaluation_policy = dict(protocol["generation_config"])
evaluation_policy["eos_token_id"] = stop_token_ids
code_hashes = {
    str(path.relative_to(PROJECT_ROOT)): sha256_file(path)
    for path in sorted((PROJECT_ROOT / "src").glob("*.py"))
}
notebook_source = json.loads(
    (PROJECT_ROOT / "notebooks/04_on_policy_distillation.ipynb").read_text()
)
# Hash the teaching/experiment code, excluding editable runtime path/mode setup.
# Output cells and Markdown edits do not change the resume identity.
core_code = [
    (cell["id"], cell["source"])
    for cell in notebook_source["cells"]
    if cell["cell_type"] == "code" and cell["id"] not in {"opd-setup-runtime", "opd-configuration"}
]
code_hashes["teacher_chat_template"] = teacher_template_sha256
# Guard thresholds are part of the signed experiment, alongside the loss settings.
code_hashes["training_safety_policy"] = hashlib.sha256(
    json.dumps(
        asdict(safety_policy) if safety_policy is not None else None, sort_keys=True
    ).encode()
).hexdigest()
code_hashes["notebook04_core_code"] = hashlib.sha256(
    json.dumps(core_code, ensure_ascii=False).encode()
).hexdigest()
signature = experiment_signature(
    config, protocol, splits, checkpoint_hashes, evaluation_policy, code_hashes
)
print("Shared tokenizer verified:", tokenizer_checks)
print("04 stopping IDs:", stop_token_ids)
print("Evaluation: greedy | max new tokens:", evaluation_policy["max_new_tokens"])

if MODE == "train":
    device, compute_dtype = torch.device("cuda:0"), torch.bfloat16
    run_directory = (
        Path(RESUME_RUN_DIRECTORY) if RESUME_RUN_DIRECTORY else new_run_directory(PROJECT_ROOT)
    )
    if RUN_STAGE == "full" and config.eos_loss_mode == "semantic_stop":
        gate_path = run_directory.parent.parent / "eos_comparison_gates.json"
        assert gate_path.is_file(), "Run and review the matched EOS comparison first."
        comparison_gates_record = json.loads(gate_path.read_text())
        assert comparison_gates_record["ready_for_full_run"], (
            "Review a passing EOS comparison before continuing the semantic pilot."
        )
        assert comparison_gates_record["arm_signatures"]["semantic_stop"] == signature, (
            "The passing comparison belongs to a different experiment."
        )
    specification_path = run_directory / "experiment.json"
    if specification_path.exists():
        assert json.loads(specification_path.read_text())["signature"] == signature, (
            "Resume protocol differs. Start a new run."
        )
    else:
        write_json(
            specification_path,
            {
                "signature": signature,
                "configuration": asdict(config),
                "protocol": protocol,
                "teacher_model_id": TEACHER_ID,
                "teacher_revision": TEACHER_REVISION,
                "evaluation_policy": evaluation_policy,
                "stop_token_ids": stop_token_ids,
                "code_hashes": code_hashes,
                "teacher_context": config.teacher_context,
                "eos_loss_mode": config.eos_loss_mode,
                "eos_token_ids": config.eos_token_ids,
                "teacher_template_sha256": teacher_template_sha256,
                "sampling_policy": {
                    "do_sample": True,
                    "temperature": 1.0,
                    "top_k": 0,
                    "top_p": 1.0,
                    "max_new_tokens": config.max_response_length,
                    "eos_token_id": stop_token_ids,
                    "use_model_defaults": False,
                },
                "evaluation_use_model_defaults": False,
                "starting_checkpoint_hashes": checkpoint_hashes,
                "tokenizer_checks": tokenizer_checks,
                "historical_run_directory": str(HISTORICAL_RUN_DIRECTORY)
                if HISTORICAL_RUN_DIRECTORY is not None
                else None,
            },
        )
        write_json(run_directory / "data_splits.json", splits)
        write_json(
            run_directory / "cold_start_evidence.json",
            {
                "manifest": sft_evidence["manifest"],
                "checkpoint_hashes": checkpoint_hashes,
                "configuration": json.loads(
                    (sft_report_directory / "configuration.json").read_text()
                ),
                "original_sft_splits": cold_splits,
                "report_checksums": json.loads(
                    (sft_report_directory / "SHA256SUMS.json").read_text()
                ),
            },
        )
    write_json(run_directory / "manifest.json", {"status": "in_progress", "signature": signature})
    print("04 artifacts on the runtime:", run_directory)
else:
    print(
        "INSPECT: source identity, new split isolation, full token mapping, and encoding checks passed."
    )
    inspection_directory = PROJECT_ROOT / "results/notebook04_qa"
    inspection_directory.mkdir(parents=True, exist_ok=True)
    write_json(
        inspection_directory / "inspection_facts.json",
        {
            "mode": "inspect",
            "split_counts": {
                name: len(splits[name])
                for name in ("train", "validation", "primary_test", "historical_test")
            },
            "selection_exclusions": len(splits["exclusions"]),
            "tokenizer_checks": tokenizer_checks,
            "stop_token_ids": stop_token_ids,
            "teacher_revision": TEACHER_REVISION,
            "maximum_selected_reference_length": max(
                len(row["input_ids"]) for row in train_examples
            ),
            "maximum_selected_prompt_length": max(row["prompt_length"] for row in train_examples),
            "model_weights_loaded": False,
        },
    )

Shared tokenizer verified: {'identical_token_mapping': True, 'text_probes_checked': 2180, 'output_vocabulary_size': 151936, 'student_eos_token_id': 151643, 'teacher_eos_token_id': 151645}
04 stopping IDs: [151643, 151645]
Evaluation: greedy | max new tokens: 512
04 artifacts on the runtime: /cache/home/igg11/on-policy-distillation/results/on_policy_distillation_20261116T022121085005Z


In [11]:
# These helpers cache completed evaluations only inside this fixed-signature run.
# They retain every response, token ID, stop flag, and frozen checker result.
def evaluate_and_save(model, examples, name, *, directory=None, identity=None, budget=None):
    # Each completed question is saved immediately. On a new allocation, matching
    # cached rows are checked and only the remaining questions are generated.
    return evaluate_incrementally(
        model,
        tokenizer,
        examples,
        evaluation_policy,
        device,
        compute_dtype,
        path=Path(directory or run_directory) / f"{name}.json",
        identity=f"{identity or signature}:{name}",
        budget=budget or SlurmBudget.from_environment(),
    )


if MODE == "train":
    starting_student = load_student(starting_checkpoint_directory, device)
    starting_validation_loss = validation_loss(
        starting_student, validation_examples, tokenizer.pad_token_id, device, compute_dtype
    )
    starting_validation = evaluate_and_save(
        starting_student, validation_examples, "starting_sft_validation"
    )
    display(pd.DataFrame([evaluation_summary(starting_validation, "starting_sft", "validation")]))
    print("Starting validation reference-token NLL:", starting_validation_loss)
    write_json(
        run_directory / "starting_sft_metrics.json",
        {
            "validation_reference_nll": starting_validation_loss,
            "validation": evaluation_summary(starting_validation, "starting_sft", "validation"),
        },
    )
    # A separate seed and forked RNG make fixed diagnostic responses reproducible
    # without consuming the random stream later used for on-policy training.
    diagnostic_path = run_directory / "fixed_validation_trajectories.json"
    if diagnostic_path.exists():
        diagnostic_trajectories = json.loads(diagnostic_path.read_text())
    else:
        with torch.random.fork_rng(devices=[device.index]):
            torch.manual_seed(config.seed + 10000)
            diagnostic_trajectories = [
                sample_trajectory(
                    starting_student,
                    tokenizer,
                    row,
                    device=device,
                    dtype=compute_dtype,
                    max_new_tokens=128,
                    stop_token_ids=stop_token_ids,
                    policy_step=0,
                )
                for row in validation_examples[: config.diagnostic_examples]
            ]
        write_json(diagnostic_path, diagnostic_trajectories)
    # Fixed validation trajectories are diagnostics only. They never enter training.
    del starting_student
    release_cuda_memory()
else:
    print("INSPECT: no baseline generation, model loss, or GPU performance measured.")

starting_sft_validation: evaluating 1/64
starting_sft_validation: evaluating 16/64
starting_sft_validation: evaluating 32/64
starting_sft_validation: evaluating 48/64
starting_sft_validation: evaluating 64/64


,stage,split,questions,strict_accuracy,format_compliance,prose_numeric_accuracy,prose_parse_rate,token_limit_stops
0,starting_sft,validation,64,0.515625,0.953125,0.5,0.90625,3


Starting validation reference-token NLL: 0.6676512348477289


---
<a id="control"></a>
## 6. Prepare the comparison and SFT control

The default fresh comparison needs your verified 03 checkpoint and matching report.
Both EOS pilots begin from those same weights. Their shared baseline is measured
in Section 5. This short comparison tests whether grouping ending probabilities
stabilizes learning while preserving task performance.

The earlier Amarel SFT control reached 40/64 strict validation matches. That is
historical evidence. For a later full experiment, this fresh workflow trains a
new 128-update SFT control for a paired evaluation on the reserved test questions.
You can optionally set `HISTORICAL_RUN_DIRECTORY` to verify and reuse the old
control and investigate its saved student checkpoints.

The visible loop below serves both training methods. An optimizer update changes
student weights after eight examples contribute gradients. All eight on-policy
responses come from the same policy version before the update.

Defaults use FP32 student weights, BF16 autocast, gradient checkpointing, AdamW at
`5e-6`, weight decay `0.01`, clipping at `1.0`, and a 128-update linear schedule
with 10% warmup. The first 32 KD updates have checks every eight updates. Each arm
retains its two latest recovery snapshots.


In [12]:
def train_arm(
    arm,
    teacher=None,
    *,
    stop_after_steps=None,
    run_config=None,
    output_directory=None,
    run_signature=None,
    recover_pause_validation=False,
):
    from dataclasses import asdict
    from datetime import datetime, timezone

    from transformers import GenerationConfig

    from src.opd_experiment import (
        SlurmBudget,
        commit_boundary,
        isolated_evaluation_rng,
        publish_pause,
    )
    from src.opd_experiment import (
        WalltimePause as TrainingWalltimePause,
    )

    # Explicit overrides keep comparison arms independent. Existing pilot/full
    # calls use the notebook's selected recipe and run directory.
    config = run_config if run_config is not None else globals()["config"]
    run_directory = (
        Path(output_directory) if output_directory is not None else globals()["run_directory"]
    )
    signature = run_signature if run_signature is not None else globals()["signature"]
    from src.training_safety import (
        accept_checkpoint,
        capture_update_rng,
        pin_starting_checkpoint,
        record_training_failure,
        restore_boundary_measurements,
        restore_update_rng,
        save_boundary_measurements,
    )

    # The new policy qualifies KD pilots. The fixed supervised control keeps
    # its existing recipe and cannot overwrite a KD accepted-checkpoint pointer.
    safety_policy = globals().get("safety_policy") if arm == "on_policy" else None
    baseline_accuracy = (
        globals().get("starting_validation", {}).get("scores", {}).get("strict", {}).get("accuracy")
    )

    def assess_health():
        if safety_policy is not None or arm == "on_policy":
            return training_health(
                history,
                config,
                starting_validation_loss,
                safety_policy=safety_policy,
                baseline_accuracy=baseline_accuracy,
            )
        return {"pause": False, "warnings": []}

    def pause_kind(assessment):
        if any(r["metric"] != "recent_cap_fraction" for r in assessment.get("reasons", [])):
            return "quality_paused"
        return "health_paused"

    budget = SlurmBudget.from_environment()
    assert arm in {"sft_control", "on_policy"}
    if arm == "on_policy" and teacher is None:
        raise ValueError("On-policy KD needs the frozen teacher.")
    target_steps = config.optimizer_steps if stop_after_steps is None else stop_after_steps
    if type(target_steps) is not int or not 1 <= target_steps <= config.optimizer_steps:
        raise ValueError("The requested stage must end within the fixed experiment.")
    final_directory = run_directory / "checkpoints" / arm
    result_path = run_directory / f"{arm}_training.json"
    progress_path = run_directory / "progress.jsonl"

    def record_progress(update, stage, event="started", **details):
        # Close each entry before starting the next operation so it can be
        # read while the run is active or after the kernel is interrupted.
        entry = {
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "arm": arm,
            "update": update,
            "stage": stage,
            "event": event,
            **details,
        }
        with progress_path.open("a", encoding="utf-8") as log:
            log.write(json.dumps(entry) + "\n")

    if final_directory.exists() and result_path.exists():
        verify_inventory(final_directory)
        completed = json.loads(result_path.read_text())
        assert completed["completed_steps"] == config.optimizer_steps
        assert completed.get("signature", signature) == signature
        print(f"{arm}: reusing its verified completed checkpoint.")
        return completed

    snapshots = sorted((run_directory / "resume" / arm).glob("step_*"))
    resume_path = snapshots[-1] if snapshots else None
    resume_metadata = (
        read_resume(resume_path, arm=arm, signature=signature) if resume_path else None
    )
    history = resume_metadata["history"] if resume_metadata else []
    completed_steps = len(history)
    if safety_policy is not None:
        if resume_metadata and resume_metadata.get("training_safety_policy") != asdict(
            safety_policy
        ):
            raise ValueError("Resume safety settings differ. Start a new experiment.")
        pin_starting_checkpoint(
            run_directory, starting_checkpoint_directory, signature, safety_policy
        )
        if resume_path is not None:
            restore_boundary_measurements(
                run_directory, history, resume_path, signature, safety_policy
            )
    if completed_steps > target_steps:
        raise ValueError("This snapshot is beyond the pilot boundary. Choose the full stage.")
    health = assess_health()
    if health["pause"] and not recover_pause_validation:
        raise RuntimeError(
            "This snapshot was health-paused. Review the evidence and start a revised experiment."
        )
    model_source = resume_path / "student" if resume_path else starting_checkpoint_directory
    student = load_student(model_source, device)
    student.generation_config = GenerationConfig.from_dict(evaluation_policy)
    student.generation_config.use_cache = True
    # These fields travel with every newly saved Hugging Face checkpoint.
    student.config.opd_eos_loss_mode = config.eos_loss_mode
    student.config.opd_eos_token_ids = list(config.eos_token_ids)

    def measure_boundary(student, record, name, *, generate_answers=True):
        # A checksummed optimizer snapshot precedes this measurement. Metrics
        # live in an atomic sidecar because the snapshot itself stays immutable.
        boundary_snapshot = run_directory / "resume" / arm / f"step_{record['step']:04d}"
        if safety_policy is not None:
            restore_boundary_measurements(
                run_directory, history, boundary_snapshot, signature, safety_policy
            )
            if "generated_validation" in record:
                return evaluate_and_save(
                    student,
                    validation_examples,
                    name,
                    directory=run_directory,
                    identity=signature,
                    budget=budget,
                )
        # Greedy validation and diagnostic forwards preserve rollout RNG states.
        with isolated_evaluation_rng():
            budget.require_time()
            record_progress(record["step"], "validation")
            record["validation_reference_nll"] = validation_loss(
                student, validation_examples, tokenizer.pad_token_id, device, compute_dtype
            )
            if safety_policy is not None:
                save_boundary_measurements(
                    run_directory,
                    record,
                    boundary_snapshot,
                    signature,
                    safety_policy,
                    complete=False,
                )
                assessment = assess_health()
                if assessment["pause"]:
                    publish_pause(
                        run_directory,
                        arm=arm,
                        status=pause_kind(assessment),
                        step=len(history),
                        snapshot=boundary_snapshot,
                        signature=signature,
                        config=config,
                        health=assessment,
                        reason="validation quality threshold",
                    )
            if teacher is not None:
                budget.require_time()
                diagnostics = fixed_trajectory_diagnostics(
                    student, teacher, diagnostic_trajectories, device, compute_dtype, config
                )
                record["fixed_validation_kl"] = diagnostics["mean_forward_kl_nats"]
                record["fixed_raw_token_kl"] = diagnostics["mean_raw_token_kl_nats"]
                record["fixed_semantic_stop_kl"] = diagnostics["mean_semantic_stop_kl_nats"]
            if not generate_answers:
                record_progress(record["step"], "validation", "completed")
                return None
            measured = evaluate_and_save(
                student,
                validation_examples,
                name,
                directory=run_directory,
                identity=signature,
                budget=budget,
            )
        record["generated_validation"] = evaluation_summary(measured, arm, "validation")
        if safety_policy is not None:
            save_boundary_measurements(
                run_directory, record, boundary_snapshot, signature, safety_policy, complete=True
            )
            if not assess_health()["pause"]:
                accept_checkpoint(
                    run_directory, boundary_snapshot, record, signature, safety_policy
                )
        record_progress(record["step"], "validation", "completed")
        return measured

    def finish_pause(student, status, reason, snapshot):
        # Publish status BEFORE validation. A timeout can leave validation pending
        # while the optimizer boundary and reason remain inspectable on disk.
        health = assess_health()
        result = publish_pause(
            run_directory,
            arm=arm,
            status=status,
            step=len(history),
            snapshot=snapshot,
            signature=signature,
            config=config,
            health=health,
            reason=reason,
        )
        result.update(history=history, resumed_from_step=completed_steps)
        if history:
            try:
                measured = measure_boundary(
                    student, history[-1], f"{arm}_boundary_{len(history):04d}_validation"
                )
                result.update(
                    validation_status="completed",
                    final_validation=evaluation_summary(measured, status, "validation"),
                    final_validation_reference_nll=history[-1]["validation_reference_nll"],
                )
            except TrainingWalltimePause:
                print(
                    "Checkpoint saved; validation will resume in the next allocation.", flush=True
                )
        result["health"] = assess_health()
        if safety_policy is not None and result["health"]["pause"]:
            status = pause_kind(result["health"])
            result["status"] = status
            result["pause_reason"] = "training or validation safety threshold"
        write_json(run_directory / f"{arm}_history.json", history)
        write_json(run_directory / f"{arm}_pause.json", result)
        write_json(run_directory / f"{arm}_{status}.json", result)
        write_json(run_directory / "manifest.json", result)
        # Publish the small report here so a disconnected notebook client does
        # not have to execute the later export cell to retain this boundary.
        if safety_policy is not None:
            try:
                export_report(
                    PROJECT_ROOT,
                    run_directory,
                    result,
                    report_name=f"report_{status}_{len(history):04d}_{time.time_ns()}",
                )
            except OSError as error:
                # The checkpoint and pause manifest already exist. A full disk
                # or transfer error must remain visible without hiding that state.
                print(
                    f"Report packaging failed. Checkpoint retained at {snapshot}. {error}",
                    flush=True,
                )
        return result

    trajectory_path = run_directory / "training_trajectories.jsonl"
    if arm == "on_policy":
        truncate_uncommitted_trajectories(trajectory_path, completed_steps)
        validate_trajectory_log(
            trajectory_path, train_examples, completed_steps, config.accumulation_steps
        )

    # A comparison may finish interrupted evaluation of an already stopped arm.
    # It never restarts updates from a health-paused policy.
    if recover_pause_validation and (health["pause"] or completed_steps == target_steps):
        try:
            return finish_pause(
                student,
                pause_kind(health) if health["pause"] else "pilot_complete",
                "health guard" if health["pause"] else "pilot boundary",
                resume_path,
            )
        finally:
            del student
            release_cuda_memory()

    seed_everything(config.seed)
    optimizer = torch.optim.AdamW(
        student.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay
    )
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=math.ceil(config.optimizer_steps * config.warmup_fraction),
        num_training_steps=config.optimizer_steps,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=False)
    if resume_path:
        restore_training_state(resume_path, optimizer, scheduler, scaler)
    if teacher is not None:
        optimizer_ids = {id(p) for group in optimizer.param_groups for p in group["params"]}
        assert not optimizer_ids.intersection(id(p) for p in teacher.parameters())
        assert all(not p.requires_grad and p.grad is None for p in teacher.parameters())
    optimizer.zero_grad(set_to_none=True)
    if torch.device(device).type == "cuda":
        for index in range(torch.cuda.device_count()):
            torch.cuda.reset_peak_memory_stats(index)
        torch.cuda.synchronize()
    started = time.perf_counter()
    pause_status, pause_reason = None, None
    latest_snapshot = resume_path
    update_pending = False
    update_rng = None

    def save_boundary(student, optimizer, scheduler, scaler, *, status=None, reason=None):
        # At step zero, the verified 03 checkpoint already supplies the start state.
        snapshot = None
        if history:
            record_progress(len(history), "checkpoint")
            snapshot = commit_boundary(
                run_directory,
                student,
                tokenizer,
                optimizer,
                scheduler,
                scaler,
                arm=arm,
                step=len(history),
                signature=signature,
                history=history,
                config=config,
                training_safety_policy=safety_policy,
            )
        if status is not None:
            # Publish immediately after the immutable checkpoint becomes visible.
            # Cleanup or later validation can then be interrupted safely.
            publish_pause(
                run_directory,
                arm=arm,
                status=status,
                step=len(history),
                snapshot=snapshot,
                signature=signature,
                config=config,
                health=health,
                reason=reason,
            )
        if history:
            record_progress(len(history), "checkpoint", "completed")
        return snapshot

    try:
        # A boundary snapshot can precede its validation. Finish that work before
        # generating the next group, using cached answers from interrupted evaluation.
        if history and "generated_validation" not in history[-1]:
            measure_boundary(student, history[-1], f"{arm}_boundary_{len(history):04d}_validation")
        for update_index in range(completed_steps + 1, target_steps + 1):
            # Re-evaluate recovered validation BEFORE allowing another update.
            health = assess_health()
            if health["pause"]:
                pause_status, pause_reason = (
                    pause_kind(health),
                    "saved validation or stopping threshold",
                )
                latest_snapshot = save_boundary(
                    student, optimizer, scheduler, scaler, status=pause_status, reason=pause_reason
                )
                break
            budget.require_time()
            # If the deadline arrives mid-group, restore the RNG from this
            # clean boundary before saving the previous completed update.
            update_rng = capture_update_rng() if safety_policy is not None else None
            update_pending = True
            group_start = (update_index - 1) * config.accumulation_steps
            examples = train_examples[group_start : group_start + config.accumulation_steps]
            step_started = time.perf_counter()
            student.train()
            if arm == "on_policy":
                # All eight fresh responses come from the same unchanged policy.
                record_progress(update_index, "student_generation")
                trajectories = []
                for example_index, example in enumerate(examples):
                    if safety_policy is not None:
                        budget.require_time()
                        record_progress(
                            update_index,
                            "student_generation",
                            example_index=example_index,
                            example_id=example["example_id"],
                        )
                    trajectories.append(
                        sample_trajectory(
                            student,
                            tokenizer,
                            example,
                            device=device,
                            dtype=compute_dtype,
                            max_new_tokens=config.max_response_length,
                            stop_token_ids=stop_token_ids,
                            policy_step=update_index - 1,
                        )
                    )
                record_progress(update_index, "teacher_scoring_and_backward")
                metrics = backward_opd_group(
                    student,
                    teacher,
                    trajectories,
                    device,
                    compute_dtype,
                    config,
                    scaler,
                    policy_step=update_index - 1,
                    before_trajectory=budget.require_time if safety_policy is not None else None,
                )
            else:
                # Reference response labels receive cross-entropy. Prompt/padding
                # labels are -100, preserving real EOS tokens as training targets.
                record_progress(update_index, "supervised_forward_and_backward")
                metrics = backward_sft_group(
                    student, examples, tokenizer.pad_token_id, device, compute_dtype, scaler
                )
            record_progress(update_index, "optimizer_step")
            update_metrics = optimizer_update(
                student, optimizer, scheduler, scaler, config.max_grad_norm
            )
            if torch.device(device).type == "cuda":
                torch.cuda.synchronize()
            record = {
                "step": update_index,
                **metrics,
                **update_metrics,
                "update_seconds": time.perf_counter() - step_started,
                "peak_allocated_gpu_gib_so_far": torch.cuda.max_memory_allocated() / 2**30
                if torch.device(device).type == "cuda"
                else None,
            }
            if torch.device(device).type == "cuda":
                record["gpu_memory"] = [
                    {
                        "index": index,
                        "peak_allocated_gib": torch.cuda.max_memory_allocated(index) / 2**30,
                    }
                    for index in range(torch.cuda.device_count())
                ]
            if arm == "on_policy":
                append_trajectories(trajectory_path, trajectories)
                del trajectories
            history.append(record)
            update_pending = False
            health = assess_health()
            for warning in health["warnings"]:
                print("REVIEW:", warning, flush=True)
            write_json(run_directory / f"{arm}_history.json", history)
            record_progress(update_index, "training_update", "completed")
            print(
                f"{arm} [{config.eos_loss_mode}] | update {update_index:3d}/{config.optimizer_steps} | "
                f"loss {metrics['loss']:.4f} | mean response tokens {metrics['response_tokens'] / metrics['examples']:.1f} | "
                f"cap stops {metrics['token_limit_stops']}/{metrics['examples']} | EOS stops {metrics.get('eos_stops', 'reference')}",
                flush=True,
            )
            interval = (
                config.pilot_check_interval
                if arm == "on_policy" and update_index <= config.pilot_updates
                else config.checkpoint_interval
            )
            validation_interval = (
                config.pilot_check_interval
                if update_index <= config.pilot_updates
                else config.validation_interval
            )
            boundary = (
                update_index % interval == 0 or update_index == target_steps or health["pause"]
            )
            if health["pause"]:
                pause_status, pause_reason = pause_kind(health), "training safety threshold"
            elif update_index == target_steps and target_steps < config.optimizer_steps:
                pause_status, pause_reason = "pilot_complete", "pilot boundary"
            if boundary:
                latest_snapshot = save_boundary(
                    student, optimizer, scheduler, scaler, status=pause_status, reason=pause_reason
                )
            if pause_status is not None:
                # finish_pause saves each follow-up answer incrementally.
                break
            if update_index % validation_interval == 0 or update_index == target_steps:
                # Save BEFORE the potentially slow generation of validation answers.
                if not boundary:
                    latest_snapshot = save_boundary(student, optimizer, scheduler, scaler)
                measure_boundary(
                    student,
                    record,
                    f"{arm}_boundary_{update_index:04d}_validation",
                    generate_answers=arm == "on_policy" or update_index == target_steps,
                )
                write_json(run_directory / f"{arm}_history.json", history)
                # Fresh validation can pause immediately, before the next update.
                health = assess_health()
                if health["pause"]:
                    pause_status, pause_reason = pause_kind(health), "validation quality threshold"
                    latest_snapshot = save_boundary(
                        student,
                        optimizer,
                        scheduler,
                        scaler,
                        status=pause_status,
                        reason=pause_reason,
                    )
                    break
    except TrainingWalltimePause:
        if update_pending and update_rng is not None:
            optimizer.zero_grad(set_to_none=True)
            restore_update_rng(update_rng)
        pause_status, pause_reason = "walltime_paused", "SLURM checkpoint reserve"
        latest_snapshot = save_boundary(
            student, optimizer, scheduler, scaler, status=pause_status, reason=pause_reason
        )
    except (KeyboardInterrupt, RuntimeError, FloatingPointError, OSError) as error:
        # An incomplete update is never published as a valid recovery snapshot.
        # Existing complete boundaries remain available even if storage is full.
        try:
            failure = record_training_failure(
                run_directory,
                error=error,
                step=len(history),
                snapshot=latest_snapshot,
                signature=signature,
            )
            if safety_policy is not None:
                export_report(
                    PROJECT_ROOT,
                    run_directory,
                    failure,
                    report_name=f"report_failure_{len(history):04d}_{time.time_ns()}",
                )
        except (OSError, ValueError) as export_error:
            print("Failure-report export could not finish:", export_error, flush=True)
        del optimizer, scheduler, scaler, student
        release_cuda_memory()
        raise
    elapsed = time.perf_counter() - started
    peak_gib = (
        torch.cuda.max_memory_allocated() / 2**30 if torch.device(device).type == "cuda" else None
    )
    del optimizer, scheduler, scaler
    release_cuda_memory()
    if arm == "on_policy":
        validate_trajectory_log(
            trajectory_path, train_examples, len(history), config.accumulation_steps
        )
    if pause_status is not None or len(history) < config.optimizer_steps:
        status = pause_status or "pilot_complete"
        try:
            return finish_pause(student, status, pause_reason or "pilot boundary", latest_snapshot)
        finally:
            del student
            release_cuda_memory()

    record_progress(len(history), "final_checkpoint")
    final_directory.parent.mkdir(parents=True, exist_ok=True)
    if final_directory.exists():
        verify_inventory(final_directory)
    else:
        save_checkpoint(student, tokenizer, final_directory)
    checkpoint_verified = verify_checkpoint(
        student,
        tokenizer,
        final_directory,
        validation_examples[0]["input_ids"],
        device,
        compute_dtype,
        probe_texts=["  spaces\n", "3 + 2 = 5.\n#### 5", "Café 中文 🙂"],
    )
    record_progress(len(history), "final_checkpoint", "completed")
    record_progress(len(history), "validation")
    final_validation = evaluate_and_save(
        student,
        validation_examples,
        f"{arm}_boundary_{len(history):04d}_validation",
        directory=run_directory,
        identity=signature,
        budget=budget,
    )
    result = {
        "arm": arm,
        "status": "completed",
        "signature": signature,
        "eos_loss_mode": config.eos_loss_mode,
        "validation_status": "completed",
        "health": assess_health(),
        "completed_steps": len(history),
        "history": history,
        "current_session_loop_seconds": elapsed,
        "peak_allocated_gpu_gib": peak_gib,
        "summed_recorded_update_seconds": sum(row["update_seconds"] for row in history),
        "maximum_recorded_peak_gpu_gib": max(
            (
                row["peak_allocated_gpu_gib_so_far"]
                for row in history
                if row["peak_allocated_gpu_gib_so_far"] is not None
            ),
            default=None,
        ),
        "checkpoint_reload_verified": checkpoint_verified,
        "final_validation": evaluation_summary(final_validation, arm, "validation"),
        "final_validation_reference_nll": history[-1]["validation_reference_nll"],
        "resumed_from_step": completed_steps,
    }
    write_json(result_path, result)
    record_progress(len(history), "validation", "completed")
    del student
    release_cuda_memory()
    return result

In [13]:
# A fresh EOS comparison trains two KD pilots from 03 and measures their shared
# starting student. A full experiment also needs a trained SFT control.
control_result = None
historical_evidence = {"status": "fresh_comparison", "starting_point": "verified_03"}
if MODE == "train":
    if HISTORICAL_RUN_DIRECTORY is not None and RUN_STAGE in {"eos_comparison", "full"}:
        historical_evidence = verify_historical_run(
            HISTORICAL_RUN_DIRECTORY, checkpoint_hashes, protocol, splits, config
        )
        control_result = historical_evidence["control"]
        control_checkpoint_directory = Path(historical_evidence["control_checkpoint"])
        print("Historical SFT control from:", historical_evidence["directory"])
    elif RUN_STAGE != "eos_comparison":
        # Build this control only when continuing to a full experiment or using
        # the standalone pilot workflow. It uses the same fixed training recipe.
        control_result = train_arm("sft_control")
        control_checkpoint_directory = run_directory / "checkpoints/sft_control"
    else:
        print("Fresh comparison: both EOS pilots start from the verified 03 checkpoint.")
    write_json(run_directory / "historical_evidence.json", historical_evidence)
    if control_result is not None:
        display(pd.DataFrame([control_result["final_validation"]]))
else:
    print("INSPECT: comparison orchestration defined; GPU arms remain pending.")

Fresh comparison: both EOS pilots start from the verified 03 checkpoint.


---
<a id="teacher"></a>
## 7. Qualify the teacher and prove response alignment

The teacher receives its non-thinking chat question context, followed by the
**exact response IDs the student generated**. Its prefix length is $P_T$, while the
student's is $P_S$. Their first-response predictions occur at $P_T-1$ and $P_S-1$,
respectively. The aligned response-logit tensors both have shape $[1,R,V]$ even
when their complete-sequence lengths differ.

For example, the same response IDs for `3 + 2 = 5` follow a plain `Answer:` prefix
for the student and an assistant-turn prefix for the teacher. We append those IDs
directly, avoiding changed token boundaries from decoding and retokenizing them.

We also generate teacher answers on the **64 validation questions** to measure
its format, numeric matching, and stopping behavior under this context. These
answers are never substituted into distillation training. Greater model size and
higher stopping probability alone do not prove useful arithmetic supervision.

`eval()` selects inference behavior. `requires_grad_(False)` freezes parameters,
and `torch.no_grad()` prevents a teacher gradient graph. KV cache is disabled for
scoring known sequences and enabled for autoregressive evaluation.


In [14]:
if MODE == "train":
    print(
        "Loading the frozen 32B teacher, about 61 GiB of weights. The first download can take several minutes.",
        flush=True,
    )
    teacher = load_teacher(device)
    # The frozen teacher may span two GPUs. Never call teacher.to(device),
    # which would collapse its layer placement onto one card.
    runtime["teacher_device_map"] = {
        str(k): str(v) for k, v in getattr(teacher, "hf_device_map", {"": str(device)}).items()
    }
    # Teacher-generated answers are an independent validation sanity check.
    teacher_validation_examples = []
    for row in validation_examples:
        teacher_prompt = teacher_tokenizer.decode(
            row["teacher_prompt_token_ids"], skip_special_tokens=False
        )
        assert (
            tokenizer.encode(teacher_prompt, add_special_tokens=False)
            == row["teacher_prompt_token_ids"]
        )
        teacher_validation_examples.append({**row, "prompt": teacher_prompt})
    teacher_validation = evaluate_and_save(
        teacher, teacher_validation_examples, "teacher_validation"
    )
    display(pd.DataFrame([evaluation_summary(teacher_validation, "teacher", "validation")]))
    if (
        teacher_validation["scores"]["strict"]["accuracy"]
        <= starting_validation["scores"]["strict"]["accuracy"]
    ):
        print("REVIEW: the teacher did not exceed starting-student strict validation accuracy.")
    student = load_student(starting_checkpoint_directory, device)
    student.eval()
    trajectory = diagnostic_trajectories[0]
    complete_sequence_ids, attention_mask = trajectory_tensors(trajectory, device)
    teacher_sequence_ids, teacher_attention_mask = trajectory_tensors(
        trajectory, device, context="teacher"
    )
    prompt_length = trajectory["prompt_length"]
    teacher_prompt_length = trajectory["teacher_prompt_length"]
    response_token_ids = complete_sequence_ids[:, prompt_length:]
    response_length = response_token_ids.shape[1]
    # Each model has its own [1, complete_length] input and causal prompt offset.
    with torch.no_grad(), precision_context(device, compute_dtype):
        student_logits = student(
            input_ids=complete_sequence_ids, attention_mask=attention_mask, use_cache=False
        ).logits
        teacher_logits = teacher(
            input_ids=teacher_sequence_ids, attention_mask=teacher_attention_mask, use_cache=False
        ).logits
    assert teacher_logits.shape[:2] == teacher_sequence_ids.shape
    assert student_logits.shape[:2] == complete_sequence_ids.shape
    print(
        "Student logits [batch, student_sequence, vocabulary]:",
        tuple(student_logits.shape),
    )
    student_response_logits = response_prediction_logits(
        student_logits, complete_sequence_ids, prompt_length
    )
    teacher_response_logits = response_prediction_logits(
        teacher_logits, teacher_sequence_ids, teacher_prompt_length
    ).to(device, copy=True)
    assert student_response_logits.shape == teacher_response_logits.shape
    assert torch.equal(
        complete_sequence_ids[:, prompt_length:], teacher_sequence_ids[:, teacher_prompt_length:]
    )
    print("Teacher logits [batch, teacher_sequence, vocabulary]:", tuple(teacher_logits.shape))
    print(
        "Aligned response logits [batch, response_tokens, vocabulary]:",
        tuple(student_response_logits.shape),
    )
    # These selected scores describe the tokens that appeared. The KL below
    # separately compares ALL vocabulary choices at each response position.
    response_student_log_probs = selected_log_probs(student_response_logits, response_token_ids)
    response_teacher_target_log_probs = selected_log_probs(
        teacher_response_logits.float() / config.teacher_temperature, response_token_ids
    )
    assert (
        response_student_log_probs.shape
        == response_teacher_target_log_probs.shape
        == (1, response_length)
    )
    assert torch.equal(complete_sequence_ids[:, 1:][:, prompt_length - 1 :], response_token_ids)
    assert (
        torch.isfinite(student_response_logits).all()
        and torch.isfinite(teacher_response_logits).all()
    )
    assert (
        torch.isfinite(response_student_log_probs).all() and (response_student_log_probs <= 0).all()
    )
    assert (
        torch.isfinite(response_teacher_target_log_probs).all()
        and (response_teacher_target_log_probs <= 0).all()
    )
    smoke_kl = forward_kl_loss(
        student_response_logits,
        teacher_response_logits,
        teacher_temperature=config.teacher_temperature,
        chunk_size=config.kl_chunk_size,
        recompute=False,
    )
    print("One unchanged diagnostic response: raw-token forward KL =", float(smoke_kl), "nats")
    display(
        pd.DataFrame(
            [
                {
                    "final_prompt_token": repr(
                        tokenizer.decode([complete_sequence_ids[0, prompt_length - 1].item()])
                    ),
                    "first_response_token": repr(
                        tokenizer.decode([response_token_ids[0, 0].item()])
                    ),
                    "first_response_input_position": prompt_length,
                    "student_predicting_position": prompt_length - 1,
                    "teacher_predicting_position": teacher_prompt_length - 1,
                    "student_log_probability": response_student_log_probs[0, 0].item(),
                    "sharpened_teacher_target_log_probability": response_teacher_target_log_probs[
                        0, 0
                    ].item(),
                }
            ]
        )
    )
    display(
        pd.DataFrame(
            [
                {
                    "response_position": position,
                    "token_id": token_id,
                    "decoded_token": repr(tokenizer.decode([token_id])),
                    "student_log_probability": response_student_log_probs[0, position].item(),
                    "sharpened_teacher_target_log_probability": response_teacher_target_log_probs[
                        0, position
                    ].item(),
                }
                for position, token_id in enumerate(trajectory["response_token_ids"][:12])
            ]
        )
    )
    # At the final sampled position, inspect total ending mass across BOTH
    # recognized stop IDs. The teacher can prefer im_end over endoftext.
    # If this response was capped, this position is not a natural end boundary.
    boundary_record = {
        "sampled_ended_with_eos": trajectory["ended_with_eos"],
        "last_sampled_token": repr(tokenizer.decode([trajectory["response_token_ids"][-1]])),
        "student_stop_probability_T1": student_response_logits[0, -1]
        .float()
        .softmax(-1)[stop_token_ids]
        .sum()
        .item(),
        "teacher_stop_probability_T1": teacher_response_logits[0, -1]
        .float()
        .softmax(-1)[stop_token_ids]
        .sum()
        .item(),
        "teacher_target_stop_probability": (
            teacher_response_logits[0, -1].float() / config.teacher_temperature
        )
        .softmax(-1)[stop_token_ids]
        .sum()
        .item(),
    }
    display(pd.DataFrame([boundary_record]))
    write_json(run_directory / "teacher_boundary_probe.json", boundary_record)
    assert all(not p.requires_grad and p.grad is None for p in teacher.parameters())
    print(
        "Frozen teacher, identical response prefixes, separate causal shifts, and finite-value checks passed."
    )
    # Free the inspection tensors before training; retaining a notebook variable
    # can otherwise keep its large vocabulary tensor on the GPU.
    del student, student_logits, teacher_logits, student_response_logits, teacher_response_logits
    del response_student_log_probs, response_teacher_target_log_probs, smoke_kl
    release_cuda_memory()
    diagnostic_student = load_student(starting_checkpoint_directory, device)
    starting_fixed_diagnostics = fixed_trajectory_diagnostics(
        diagnostic_student, teacher, diagnostic_trajectories, device, compute_dtype, config
    )
    write_json(run_directory / "starting_sft_fixed_diagnostics.json", starting_fixed_diagnostics)
    del diagnostic_student
    release_cuda_memory()
else:
    print("INSPECT: teacher weights and real alignment logits still require a CUDA GPU run.")

Loading the frozen 32B teacher, about 61 GiB of weights. The first download can take several minutes.
Kernel PID 2535206: frozen BF16 teacher device map OrderedDict([('model.embed_tokens', 0), ('model.layers.0', 0), ('model.layers.1', 0), ('model.layers.2', 0), ('model.layers.3', 0), ('model.layers.4', 0), ('model.layers.5', 0), ('model.layers.6', 0), ('model.layers.7', 0), ('model.layers.8', 0), ('model.layers.9', 0), ('model.layers.10', 0), ('model.layers.11', 0), ('model.layers.12', 0), ('model.layers.13', 0), ('model.layers.14', 0), ('model.layers.15', 0), ('model.layers.16', 0), ('model.layers.17', 0), ('model.layers.18', 0), ('model.layers.19', 0), ('model.layers.20', 0), ('model.layers.21', 0), ('model.layers.22', 0), ('model.layers.23', 0), ('model.layers.24', 0), ('model.layers.25', 0), ('model.layers.26', 0), ('model.layers.27', 0), ('model.layers.28', 0), ('model.layers.29', 0), ('model.layers.30', 0), ('model.layers.31', 1), ('model.layers.32', 1), ('model.layers.33', 1), (

,stage,split,questions,strict_accuracy,format_compliance,prose_numeric_accuracy,prose_parse_rate,token_limit_stops
0,teacher,validation,64,0.828125,0.875,0.78125,0.828125,0


Student logits [batch, student_sequence, vocabulary]: (1, 159, 151936)
Teacher logits [batch, teacher_sequence, vocabulary]: (1, 171, 151936)
Aligned response logits [batch, response_tokens, vocabulary]: (1, 86, 151936)
One unchanged diagnostic response: raw-token forward KL = 1.6748522520065308 nats


,final_prompt_token,first_response_token,first_response_input_position,student_predicting_position,teacher_predicting_position,student_log_probability,sharpened_teacher_target_log_probability
0,':\n','First',73,72,84,-1.796874,-161.25


,response_position,token_id,decoded_token,student_log_probability,sharpened_teacher_target_log_probability
0,0,5338,'First',-1.796874,-161.250000
1,1,1477,' find',-0.242906,-206.875000
2,2,279,' the',-0.354242,0.000000
3,3,1372,' number',-0.953965,-0.000004
4,4,315,' of',-0.003573,0.000000
5,5,6310,' customers',-0.185665,-40.000046
6,6,879,' who',-0.072341,0.000000
7,7,10788,' bought',-0.057438,0.000000
8,8,2326,' three',-0.779295,-32.500000
9,9,3015,' water',-0.702669,-25.000000


,sampled_ended_with_eos,last_sampled_token,student_stop_probability_T1,teacher_stop_probability_T1,teacher_target_stop_probability
0,True,'<|endoftext|>',0.999987,0.909857,1.0


Frozen teacher, identical response prefixes, separate causal shifts, and finite-value checks passed.


### 7.1 Inspect stopping probabilities before training

A fresh Colab run probes the starting 03 student on its eight fixed validation
responses. For each response, we inspect individual EOS probabilities and their
combined stopping probability. We also compute how each loss would change detached
logits, while keeping model weights fixed.

A prefix containing $k$ response tokens predicts its next token at position
$P+k-1$. Before an observed EOS, that EOS is a target. After a capped response,
there is no observed next token. The probe uses the original response token IDs.

If `HISTORICAL_RUN_DIRECTORY` points to a complete old Amarel run, this section
also probes updates 16 and 22 on identical prefixes and recovers update-22
validation. The default fresh comparison begins with the 03 probe only.

`stopping_probes.json` contains every EOS probability, logit gradient, causal
position, and decoded prefix tail. The preview shows a subset of those records.


In [15]:
# Frozen probes inspect the starting student before any comparison updates.
if MODE == "train" and RUN_STAGE == "eos_comparison":
    if HISTORICAL_RUN_DIRECTORY is None:
        # Reuse original response IDs from the starting student's fixed validation
        # trajectories. The fresh run can diagnose EOS probabilities before training.
        selected_probes = [
            dict(row, probe_group="fresh_starting_validation") for row in diagnostic_trajectories
        ]
        probe_checkpoints = {"starting_03": starting_checkpoint_directory}
    else:
        selected_probes = load_historical_probes(HISTORICAL_RUN_DIRECTORY)
        probe_checkpoints = {
            "starting_03": starting_checkpoint_directory,
            "step_0016": Path(historical_evidence["snapshots"]["16"]) / "student",
            "step_0022": Path(historical_evidence["snapshots"]["22"]) / "student",
        }
    write_json(run_directory / "probe_trajectories.json", selected_probes)
    probe_path = run_directory / "stopping_probes.json"
    probe_identity = hashlib.sha256(
        json.dumps(
            {
                "signature": signature,
                "historical_signature": historical_evidence.get("signature"),
                "original_trajectories": selected_probes,
            },
            sort_keys=True,
        ).encode()
    ).hexdigest()
    if probe_path.exists():
        stopping_probes = json.loads(probe_path.read_text())
        assert stopping_probes["probe_identity"] == probe_identity, (
            "Historical prefixes or checkpoint identities changed. Start a fresh diagnosis."
        )
    else:
        stopping_probes = probe_checkpoint_set(
            teacher,
            tokenizer,
            selected_probes,
            probe_checkpoints,
            device,
            compute_dtype,
            config,
            budget=SlurmBudget.from_environment(),
        )
        stopping_probes["probe_identity"] = probe_identity
        write_json(probe_path, stopping_probes)
    # Group matching prefixes when historical checkpoints are supplied.
    # A fresh run displays the starting 03 student. The full JSON retains
    # every individual EOS probability, logit gradient, and decoded prefix tail.
    probe_table = pd.DataFrame(stopping_probes["rows"]).sort_values(
        ["probe_group", "example_id", "response_prefix_length", "checkpoint"]
    )
    display(
        probe_table[
            [
                "checkpoint",
                "example_id",
                "boundaries",
                "response_prefix_length",
                "student_predicting_position",
                "teacher_predicting_position",
                "observed_next_token",
                "student_stop_probability",
                "teacher_T1_stop_probability",
                "teacher_target_stop_probability",
                "raw_token_kl_nats",
                "semantic_stop_kl_nats",
            ]
        ].head(18)
    )
    # New probes expose target sharpness and separate EOS contributions.
    # Historical probe files remain readable when these fields were not recorded.
    extra_columns = [
        "teacher_T1_entropy_nats",
        "teacher_target_entropy_nats",
        "ordinary_token_contribution_nats",
        "raw_eos_contribution_nats",
        "grouped_eos_contribution_nats",
    ]
    if all(column in probe_table.columns for column in extra_columns):
        display(probe_table[["example_id", "response_prefix_length", *extra_columns]].head(8))
    if HISTORICAL_RUN_DIRECTORY is not None:
        # The previous allocation ended during this validation. Continue generation
        # from the saved weights, saving each answer into the NEW diagnosis folder.
        recovered_student = load_student(probe_checkpoints["step_0022"], device)
        try:
            recovered_validation = evaluate_and_save(
                recovered_student,
                validation_examples,
                "historical_step_0022_validation",
                identity=f"{historical_evidence['signature']}:step_0022",
            )
            with isolated_evaluation_rng():
                recovered_nll = validation_loss(
                    recovered_student,
                    validation_examples,
                    tokenizer.pad_token_id,
                    device,
                    compute_dtype,
                )
            write_json(
                run_directory / "historical_step_0022_metrics.json",
                {
                    "validation": evaluation_summary(
                        recovered_validation, "historical_step_0022", "validation"
                    ),
                    "reference_nll": recovered_nll,
                },
            )
        finally:
            del recovered_student
            release_cuda_memory()
    print("Stopping probes saved. Review their EOS probabilities and gradients.")
else:
    print("INSPECT: frozen-model stopping probes run on the allocated GPU.")

,checkpoint,example_id,boundaries,response_prefix_length,student_predicting_position,teacher_predicting_position,observed_next_token,student_stop_probability,teacher_T1_stop_probability,teacher_target_stop_probability,raw_token_kl_nats,semantic_stop_kl_nats
6,starting_03,gsm8k/train/102,[first_response_prediction],0,92,104,'She',1.286756e-08,6.849340e-15,0.000000e+00,0.981174,0.981174
7,starting_03,gsm8k/train/102,"[before_observed_eos, after_first_final_answer...",46,138,150,'<|endoftext|>',9.991818e-01,9.013019e-01,1.000000e+00,33.407070,0.000819
8,starting_03,gsm8k/train/1951,[first_response_prediction],0,107,119,'M',2.078934e-08,6.364373e-15,0.000000e+00,6.563838,6.563838
9,starting_03,gsm8k/train/1951,[before_last_observed_token],127,234,246,' ',2.461067e-07,2.412492e-09,0.000000e+00,0.592513,0.592513
10,starting_03,gsm8k/train/1951,[after_truncation],128,235,247,NaN,2.174173e-10,3.304328e-10,0.000000e+00,0.312296,0.312296
0,starting_03,gsm8k/train/3370,[first_response_prediction],0,72,84,'First',1.986458e-08,3.191953e-15,0.000000e+00,4.296865,4.296865
1,starting_03,gsm8k/train/3370,"[before_observed_eos, after_first_final_answer...",85,157,169,'<|endoftext|>',9.999875e-01,9.098600e-01,1.000000e+00,35.375011,0.000013
2,starting_03,gsm8k/train/4852,[first_response_prediction],0,71,83,'If',1.065618e-08,5.166223e-14,0.000000e+00,7.169650,7.169650
3,starting_03,gsm8k/train/4852,"[before_observed_eos, after_first_final_answer...",98,169,181,'<|endoftext|>',5.620416e-01,4.687636e-01,2.227001e-01,7.345727,0.240200
13,starting_03,gsm8k/train/5823,[first_response_prediction],0,69,81,'The',2.317352e-08,1.204314e-13,0.000000e+00,6.830223,6.830223


Stopping probes saved. Review their EOS probabilities and gradients.


---
<a id="training"></a>
## 8. Compare raw-token and grouped-STOP pilots

**Saved outputs below describe the earlier guard policy.** New executions use
the versioned safeguards printed in Section 2. Their GPU behavior awaits validation.

Both pilots start from the verified 03 weights, use the same training questions
and initial seed, and keep the same 128-update learning-rate schedule. Each is
limited to 32 updates. Their responses can diverge as their weights change.
The frozen teacher is shared. Each student has its own optimizer, random stream,
signature, and recovery folder.

| Setting | Both pilots |
| :--- | :--- |
| Student sampling | Temperature 1, no top-k/top-p filtering |
| Teacher targets | Temperature 0.1, non-thinking chat context |
| Optimizer | AdamW, peak learning rate `5e-6`, clip norm 1 |
| Accumulation | Eight fresh student responses per update |
| Generation limit | 256 response tokens |
| Comparison | Raw-token KL versus semantic STOP KL |

Every eight updates, we save an optimizer-boundary snapshot before validation.
Validation includes reference NLL, generated answers, and both KL definitions on
the same fixed prefixes. The guarded policy pauses above 8 capped responses in a window of
32 and applies the baseline-relative quality limits from Section 2. Pause status is written before generation of the follow-up validation answers.
A health-paused optimizer stays stopped. Its unfinished validation can be recovered.
Each pilot also appends to `progress.jsonl` before each training stage starts.
If the kernel stops mid-update, check the last entry for the operation it reached.

The review gates are deliberately explicit. The grouped pilot should reach 32
updates with finite calculations, at most 8 caps in its final 32 responses,
reference NLL no more than 10% above its starting value, and at least 32/64 strict
validation matches. A meaningful EOS contrast requires at least a 25-percentage-point
cap-rate reduction at the latest common update. These are engineering thresholds
for review, not statistical evidence of generalization.

The primary test remains reserved. If the gates fail or the contrast is unclear,
report that outcome and reconsider the hypothesis before extending training.


A complete checkpoint is saved before validation. The last acceptable checkpoint
is protected from cleanup while the two newest recovery states are retained.
Completed validation measurements are bound to that checkpoint's inventory hash.
A paused run can finish its evaluation before review. Continuing its optimizer
requires a matching accepted policy and an eligible recovery state.


In [ ]:
# The two pilot policies start independently from the same verified 03 weights.
# Only EOS loss interpretation differs. The frozen teacher is loaded once.
FULL_RUN_COMPLETE = False
if MODE == "train" and RUN_STAGE == "eos_comparison":
    assert (run_directory / "stopping_probes.json").is_file(), "Run Section 7.1 first."
    comparison_storage = check_comparison_storage(run_directory)
    print("Storage check:", comparison_storage)
    comparison_result = run_eos_comparison(
        train_arm,
        teacher,
        directory=run_directory,
        config=config,
        protocol=protocol,
        splits=splits,
        checkpoint_hashes=checkpoint_hashes,
        evaluation_policy=evaluation_policy,
        code_hashes=code_hashes,
        starting_validation_nll=starting_validation_loss,
        safety_policy=safety_policy,
        report_root=PROJECT_ROOT,
        starting_validation_accuracy=starting_validation["scores"]["strict"]["accuracy"],
    )
    display(
        pd.DataFrame(
            [
                {
                    "eos_loss_mode": mode,
                    "status": result["status"],
                    "completed_updates": result["completed_steps"],
                    "validation_status": result["validation_status"],
                    **result.get("final_validation", {}),
                }
                for mode, result in comparison_result["results"].items()
            ]
        )
    )
    print("Engineering gates:", comparison_result["gates"])
    from IPython.display import Image

    from src.opd_plots import plot_eos_comparison

    plot_eos_comparison(comparison_result["results"], starting_validation_loss, run_directory)
    display(Image(filename=str(run_directory / "eos_comparison_diagnostics.png")))
    print("Review both EOS modes, response text, and task results before a full run.")
elif MODE == "train":
    if RUN_STAGE == "full":
        snapshots = sorted((run_directory / "resume/on_policy").glob("step_*"))
        assert snapshots, "A full stage must resume a reviewed pilot snapshot."
        pilot_metadata = read_resume(snapshots[-1], arm="on_policy", signature=signature)
        assert pilot_metadata["completed_steps"] >= min(
            config.pilot_updates, config.optimizer_steps
        )
    stop_after = min(config.pilot_updates, config.optimizer_steps) if RUN_STAGE == "pilot" else None
    on_policy_result = train_arm("on_policy", teacher=teacher, stop_after_steps=stop_after)
    assert all(not p.requires_grad and p.grad is None for p in teacher.parameters())
    if on_policy_result.get("validation_status") == "completed":
        display(pd.DataFrame([on_policy_result["final_validation"]]))
    print("Run status:", on_policy_result["status"])
    print("Health review:", on_policy_result["health"])
    FULL_RUN_COMPLETE = on_policy_result["status"] == "completed"
else:
    print("INSPECT: both EOS objectives and comparison wiring are ready for GPU qualification.")

Storage check: {'filesystem_free_gib': 239943.3076171875, 'personal_quota_verified': False, 'note': 'Review HPC quota before additional large recovery snapshots.'}
on_policy [raw_token] | update   1/128 | loss 1.3176 | mean response tokens 131.8 | cap stops 0/8 | EOS stops 8
on_policy [raw_token] | update   2/128 | loss 1.7752 | mean response tokens 64.4 | cap stops 0/8 | EOS stops 8
on_policy [raw_token] | update   3/128 | loss 1.3239 | mean response tokens 88.6 | cap stops 0/8 | EOS stops 8
on_policy [raw_token] | update   4/128 | loss 1.3179 | mean response tokens 120.1 | cap stops 1/8 | EOS stops 7
on_policy [raw_token] | update   5/128 | loss 1.6856 | mean response tokens 68.6 | cap stops 0/8 | EOS stops 8
on_policy [raw_token] | update   6/128 | loss 1.4917 | mean response tokens 83.9 | cap stops 0/8 | EOS stops 8
on_policy [raw_token] | update   7/128 | loss 1.1206 | mean response tokens 104.8 | cap stops 0/8 | EOS stops 8
on_policy [raw_token] | update   8/128 | loss 1.0249 | m

---
<a id="results"></a>
## 9. Evaluate fixed final checkpoints

The EOS comparison skips this section's test-generation cells. The primary held-out
exam is opened only after a fixed 128-update run completes. Each checkpoint receives the
same questions, prompt, greedy decoding, end-token rules, token budget, and frozen
answer checkers. References are used **after generation** to score the final number.

- **Strict match** counts a correct numeric answer on the required `#### <number>` final line.
- **Format compliance** counts a parseable final line, whether its number is right or wrong.
- **Exploratory prose match** uses the unchanged broader extractor to find a matching final number.
- **Token-limit stops** count responses that exhausted the budget without a recognized end token.

We keep every question in the denominator. A higher format score alone is not
better arithmetic. Numeric agreement also does not establish valid reasoning.
Paired bootstrap intervals summarize uncertainty across these questions, conditional
on one training seed. They do not measure run-to-run training variability.

The previously inspected 32-question subset is **secondary historical context**.
Its freshly generated 04 scores use 04's stopping rule and should not be presented
as a controlled comparison against old 02/03 aggregates.


In [ ]:
if MODE == "train" and FULL_RUN_COMPLETE:
    # Score unchanged validation prefixes for the control and final KD checkpoint
    # while the teacher is still available. These are not optimizer inputs.
    fixed_diagnostics = {"starting_sft": starting_fixed_diagnostics}
    for arm in ("sft_control", "on_policy"):
        checkpoint_directory = (
            control_checkpoint_directory
            if arm == "sft_control"
            else run_directory / "checkpoints" / arm
        )
        evaluated_student = load_student(checkpoint_directory, device)
        fixed_diagnostics[arm] = fixed_trajectory_diagnostics(
            evaluated_student, teacher, diagnostic_trajectories, device, compute_dtype, config
        )
        del evaluated_student
        release_cuda_memory()
    write_json(run_directory / "fixed_diagnostics.json", fixed_diagnostics)
    display(
        pd.DataFrame(
            [
                {"checkpoint": arm, "fixed_prefix_forward_kl_nats": result["mean_forward_kl_nats"]}
                for arm, result in fixed_diagnostics.items()
            ]
        )
    )
    del teacher
    release_cuda_memory()
    print("Teacher released before the final generated-answer evaluations.")
else:
    print("PENDING: no real fixed-prefix model comparison available yet.")

In [ ]:
# Every arm receives the exact same held-out questions and decoding rules.
# The teacher is already released; no reference answer is passed to generate().
if MODE == "train" and FULL_RUN_COMPLETE:
    primary_evaluations, historical_evaluations = {}, {}
    checkpoint_directories = {
        "starting_sft": starting_checkpoint_directory,
        "sft_control": control_checkpoint_directory,
        "on_policy": run_directory / "checkpoints/on_policy",
    }
    for arm, checkpoint_directory in checkpoint_directories.items():
        evaluated_student = load_student(checkpoint_directory, device)
        primary_evaluations[arm] = evaluate_and_save(
            evaluated_student, splits["primary_test"], f"{arm}_primary_test"
        )
        historical_evaluations[arm] = evaluate_and_save(
            evaluated_student, splits["historical_test"], f"{arm}_historical_test"
        )
        del evaluated_student
        release_cuda_memory()
    primary_summary = pd.DataFrame(
        [
            evaluation_summary(result, arm, "primary_test")
            for arm, result in primary_evaluations.items()
        ]
    )
    display(primary_summary)
    print("Previously inspected subset, shown separately:")
    display(
        pd.DataFrame(
            [
                evaluation_summary(result, arm, "historical_test")
                for arm, result in historical_evaluations.items()
            ]
        )
    )
    paired_comparisons = {
        f"{first}_to_{second}": paired_accuracy_difference(
            primary_evaluations[first], primary_evaluations[second]
        )
        for first, second in [
            ("starting_sft", "sft_control"),
            ("starting_sft", "on_policy"),
            ("sft_control", "on_policy"),
        ]
    }
    display(pd.DataFrame(paired_comparisons).T)
    write_json(run_directory / "paired_comparisons.json", paired_comparisons)
    plot_run(
        {"sft_control": control_result["history"], "on_policy": on_policy_result["history"]},
        primary_evaluations,
        run_directory,
    )
    from IPython.display import Image

    display(Image(filename=str(run_directory / "training_objectives.png")))
    display(Image(filename=str(run_directory / "primary_test_comparison.png")))
else:
    print("PENDING: final exam generation and performance figures await train mode.")

### 9.1 Read answers that changed

Aggregate scores can hide regressions. The next table includes both
incorrect-to-correct and correct-to-incorrect changes. Read their actual text
before deciding whether the model learned arithmetic, formatting, or a different
way of ending its answer. An empty table can be a valid result.


In [ ]:
if MODE == "train" and FULL_RUN_COMPLETE:
    before = primary_evaluations["starting_sft"]
    after = primary_evaluations["on_policy"]
    before_rows = {r["example_id"]: r for r in before["rows"]}
    after_rows = {r["example_id"]: r for r in after["rows"]}
    before_scores = {r["example_id"]: r for r in before["annotations"]}
    changed_answers = []
    for row in after["annotations"]:
        key = row["example_id"]
        if before_scores[key]["strict_correct"] != row["strict_correct"]:
            changed_answers.append(
                {
                    "example_id": key,
                    "change": "improved" if row["strict_correct"] else "regressed",
                    "question": before_rows[key]["question"],
                    "expected_answer": before_rows[key]["expected_answer"],
                    "starting_response": before_rows[key]["response_text"],
                    "on_policy_response": after_rows[key]["response_text"],
                }
            )
    # Include both directions in the visible preview instead of cherry-picking gains.
    preview = [
        r
        for direction in ("improved", "regressed")
        for r in [v for v in changed_answers if v["change"] == direction][:2]
    ]
    display(pd.DataFrame(preview))
    write_json(run_directory / "changed_answers.json", changed_answers)
else:
    print("PENDING: no generated-answer changes to inspect.")

---
<a id="save"></a>
## 10. Save the evidence and explain the outcome

The comparison report includes both independent arm histories and their review
gates. Recovery snapshots remain in each arm's directory. The small report also
includes exact split IDs, sampled trajectories, raw evaluation
responses, loss histories, source code, configuration, model revisions, resource
measurements, and checksums. It excludes full weights and optimizer snapshots.
The final Hugging Face checkpoints are saved in separate directories and verified
by reload, tokenizer round trips, and logits comparison.
Their saved generation defaults include both recognized end-token IDs and the
greedy evaluation policy, so exported weights carry their stopping settings too.

A `/content/...` path is on **Colab's server**. Creating a ZIP there does not mean
it has downloaded to your computer. In a VS Code Colab connection, use its file
explorer or download action to copy the printed report and checkpoint paths.
Save the executed notebook too. Colab storage can disappear when the session ends.

No Google Drive or automatic large browser download is used. Set
`EXPORT_CHECKPOINT_ZIPS = True` if you want separate weight ZIPs created on the
server, then download them using VS Code.


In [ ]:
# Weight archives are optional because each full student checkpoint is large.
EXPORT_CHECKPOINT_ZIPS = False
if MODE == "train":
    recorded_runtime = {
        **runtime,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "python": sys.version,
        "dataset_fingerprints": {
            "train": source_train._fingerprint,
            "test": source_test._fingerprint,
        },
        "huggingface_packages": {
            name: importlib.metadata.version(name)
            for name in [
                "transformers",
                "datasets",
                "accelerate",
                "huggingface_hub",
                "numpy",
                "pandas",
                "matplotlib",
                "ipywidgets",
                "PyYAML",
            ]
        },
    }
if MODE == "train" and RUN_STAGE == "eos_comparison":
    arm_results = comparison_result["results"]
    completed = len(arm_results) == 2 and all(
        result.get("validation_status") == "completed" and result["status"] != "walltime_paused"
        for result in arm_results.values()
    )
    manifest = {
        "status": "comparison_complete" if completed else "comparison_incomplete",
        "signature": signature,
        "method": "controlled_eos_forward_kl",
        "configuration": asdict(config),
        "runtime": recorded_runtime,
        "comparison_modes": ["raw_token", "semantic_stop"],
        "eos_token_ids": config.eos_token_ids,
        "historical_evidence": historical_evidence,
        "engineering_gates": comparison_result["gates"],
        "scope": "One seed; primary held-out test remains pending.",
    }
    boundary = "_".join(
        f"{mode}_{arm_results[mode]['completed_steps']:04d}_{arm_results[mode]['validation_status']}"
        for mode in arm_results
    )
    report_archive = export_report(
        PROJECT_ROOT, run_directory, manifest, report_name=f"report_eos_{boundary}"
    )
    print("EOS COMPARISON REPORT:", report_archive)
    print("Retain the entire run folder, including both arms' recovery snapshots.")
elif MODE == "train":
    if FULL_RUN_COMPLETE:
        assert (
            control_result["checkpoint_reload_verified"]
            and on_policy_result["checkpoint_reload_verified"]
        )
    manifest = {
        "status": on_policy_result["status"],
        "method": "on_policy_forward_kl_with_sft_control",
        "signature": signature,
        "training_seed": config.seed,
        "student_model_id": protocol["student_model_id"],
        "student_revision": protocol["student_revision"],
        "teacher_model_id": TEACHER_ID,
        "teacher_revision": TEACHER_REVISION,
        "teacher_frozen": True,
        "teacher_temperature": config.teacher_temperature,
        "teacher_context": config.teacher_context,
        "eos_loss_mode": config.eos_loss_mode,
        "eos_token_ids": config.eos_token_ids,
        "teacher_template_sha256": teacher_template_sha256,
        "completed_on_policy_updates": on_policy_result["completed_steps"],
        "health": on_policy_result["health"],
        "starting_checkpoint_hashes": checkpoint_hashes,
        "optimizer_steps_per_arm": config.optimizer_steps,
        "runtime": recorded_runtime,
        "limitations": [
            "One training seed",
            "Custom subset, not an official leaderboard score",
            "Prose checker is exploratory",
            "Numeric agreement does not verify reasoning",
            "Different losses and compute budgets across SFT and KD",
            "Teacher and student have distinct prompt contexts and shared response IDs",
            "03/02 historical metrics used different end-token rules",
        ],
    }
    write_json(run_directory / "manifest.json", manifest)
    report_name = (
        "report"
        if FULL_RUN_COMPLETE
        else f"report_{on_policy_result['status']}_{on_policy_result['completed_steps']:04d}"
    )
    report_archive = export_report(PROJECT_ROOT, run_directory, manifest, report_name=report_name)
    print("REPORT ZIP ON RUNTIME:", report_archive)
    for arm in ("sft_control", "on_policy") if FULL_RUN_COMPLETE else ("sft_control",):
        checkpoint_directory = (
            control_checkpoint_directory
            if arm == "sft_control"
            else run_directory / "checkpoints" / arm
        )
        print(f"{arm} CHECKPOINT FOLDER ON RUNTIME:", checkpoint_directory)
        if EXPORT_CHECKPOINT_ZIPS:
            archive = verified_zip(checkpoint_directory, run_directory / f"{arm}_checkpoint.zip")
            print(f"{arm} CHECKPOINT ZIP ON RUNTIME:", archive)
    if not FULL_RUN_COMPLETE:
        print("PILOT RECOVERY SNAPSHOT:", on_policy_result["resume_snapshot"])
        print(
            "For continuation on another runtime, retain the entire run folder including resume state."
        )
    print("Export checks passed. Copy these artifacts and the executed notebook to your computer.")
else:
    print("Inspection complete. No new Qwen training was performed by this inspection execution.")

### What we achieved, in the simplest terms

**01** followed one student answer through scoring, KL, gradients, and one update.
The saved update increased KL. It demonstrated mechanics and exposed limitations.

**02** measured the original student and separated format compliance from numeric
agreement. The checkers and original responses remain inspectable.

**03** taught the student from 256 worked examples and saved a verified checkpoint.
It learned the answer format strongly. Reasoning results remained mixed.

**04** exposed a real failure in teacher feedback. The student increasingly kept
writing even as its KL fell. A frozen-teacher probe showed that chat formatting
changed stopping probabilities on one saved prefix. The corrected-context Amarel
pilot passed alignment checks and performed 22 updates, then reached the stopping
health guard. The continued-SFT control completed 128 updates and scored 40/64
strict validation matches. The new experiment compares two interpretations of EOS
tokens, with separate student policies and recoverable progress.

**Current evidence boundary** · The recorded failures, teacher probes, and completed
SFT control are user-run GPU evidence. Local CPU checks test the new grouped loss,
gradients, and recovery behavior using small random models. The new 32-update EOS
comparison and fixed full-run test remain pending on Amarel. Whether this change
stabilizes stopping or improves answers remains an experimental question.

After a successful full run, inspect strict matches, format compliance, prose
matches, capped responses, paired uncertainty, and changed answers together.
Neither teacher confidence nor a lower KL alone proves better reasoning.

### A stronger follow-up

Repeat with additional training seeds after the fixed comparison. An off-policy
KD arm using the same divergence would help isolate the effect of student-generated
prefixes. The reported bootstrap interval covers question-sampling uncertainty
conditional on one training seed.

### References

- [GKD method and arithmetic experiments](https://arxiv.org/html/2306.13649v3)
- [Pinned Qwen3-32B usage and non-thinking mode](https://huggingface.co/Qwen/Qwen3-32B/blob/9216db5781bf21249d130ec9da846c4624c16137/README.md)
- [Pinned GSM8K source](https://huggingface.co/datasets/openai/gsm8k/tree/740312add88f781978c0658806c59bc2815b9866)
- [Previous notebook](03_sft_cold_start.ipynb) · [Validation record](assets/04_on_policy_distillation/VALIDATION.md)

**The current milestone** is a controlled test of stopping behavior. The Amarel
run demonstrated teacher scoring and real updates on two GPUs, then exposed
repetition. The next comparison asks whether aligning the meaning of ending tokens
stabilizes learning. Its results are pending. Finishing answers and solving them
correctly are measured separately.
